In [0]:
%run /Configurations/Init_Scripts

## Generic Functions

In [0]:
#TestCase Description and Error Message are stored in Dictionary
TestCaseDetails = {"validateCycleCount":{"ErrorMessage":"Cycles Counts are not matched between factinvoiceaddendumdetails and factullogs",
                                          "TestCaseDescription":"Validate the P3 eligible cycle count reported in UL logs vs Rules Engine processed into invoice addendum details and consumer subscription"},
                    "validateTreatmentDates" : {"ErrorMessage":"Treatment Visit Dates are not matched between factinvoiceaddendumdetails and factullogs",
                                                "TestCaseDescription":"Validate the P3 eligible Treatment Dates reported in UL logs vs Rules Engine processed into invoice addendum details and consumer subscription"},
                    "validateExceptionProcessed":{"ErrorMessage":"Few Invoice Exceptions are not processed in Rules Engine",
                                                  "TestCaseDescription":"Validate all exceptions received from Moxie are processed into invoice addendum details table"},
                    "validatePerPatientFeeException":{"ErrorMessage":"Subscriptions are not closed for Per Patient Fee Exception",
                                                      "TestCaseDescription":"For per patient exceptions validate if the subscription has ended"},
                    "validateComments":{"ErrorMessage":"Comments was not populated Correctly in promotion.fact_invoiceaddendumdetails",
                                                                "TestCaseDescription":"Validate Comments are populated correctly in promotion.fact_invoiceaddendumdetails"},
                    "validateByPatientClassification":{"ErrorMessage":"Patient Classification was not matched between Pre validation and factinvoiceaddendumdetails",
                                                      "TestCaseDescription":"Validate the P3 eligible cycle was having consumer subscription if it has check it was Per Patient Fee/Follow Up Visit record"},
                    "validateNoPatientAssociationFee":{"ErrorMessage":"No Patient Association Fee are not matched between factinvoiceaddendumdetails and factullogs",
                                                      "TestCaseDescription":"Validate NoPatientAssociationFee Calculation"},
                    "validatePerPatientFee":{"ErrorMessage":"Pre Patient Fee are not matched between factinvoiceaddendumdetails and factullogs",
                                                      "TestCaseDescription":"Validate PerPatientFee Calculation"},
                    "validatePerCycleFee":{"ErrorMessage":"Pre Cycle Fee, Invoice Amount are not matched between factinvoiceaddendumdetails and factullogs",
                                                      "TestCaseDescription":"Validate PerCycleFee Calculation"},
                    "validateOffboardingInvoiceAmount":{"ErrorMessage":"Invoice Amount was not calculated correctly in factinvoiceaddendumdetails",
                                                      "TestCaseDescription":"Validate Offboarding and Midway Invoice Calculation"},
                    "validatePerCycleEfficacyFailFee":{"ErrorMessage":"PerCycleEfficacyFail is not calculated correctly",
                                                      "TestCaseDescription":"Validate percycleefficacyfail Calculation"}
                    }

dst_TestExecutionResult = '/mnt/silver/TestExecutionResult'

In [0]:
ULLogsTable = dbutils.widgets.get('ULLogsTable')
BufferTime = dbutils.widgets.get('BufferTime')
# MidwaySoldToID = dbutils.widgets.get('MidwaySoldToAccountId')

# MidwaySoldToId_list =  MidwaySoldToID.split(",")

In [0]:
CalendarStartDate = spark.read.table('Promotion.dim_invoicecyclemonth')\
                                .filter(col("InvoiceDate") == f'{InvoiceDate}')\
                                .select(to_date("CalendarStartTimeStamp").cast("string")).collect()[0][0]

print(CalendarStartDate)

In [0]:
def add_Prefix_ColumnName(df,prefix):
    new_cols = [col(col_name).alias(prefix+col_name) for col_name in df.columns]
    df_Source = df.select(new_cols)
    return df_Source

In [0]:
#Reading data from the tables
df_ULLogs = (spark.read.table(f"{ULLogsTable}")
                    .withColumn("CoolSculptingID", when(coalesce(col("CoolSculptingID").cast("bigint"), lit(0)) == 0, "MISSING").otherwise(col("CoolSculptingID")))
                    .withColumn("PlanType",when((col("PlanType").isin("ComprehensiveTreatmentPlan")) | (col("PlanType").isNull()) | (trim(col("PlanType")) == ""),"ComprehensiveTreatmentPlan").otherwise(col("PlanType")))
                    .withColumn("TreatmentTime",when(col("HdrStartTimeGeneratedTmstmp")>col("EventEndTmstmp"),expr("timestampdiff(MINUTE,EventEndTmstmp, HdrStartTimeGeneratedTmstmp)"))
                                .otherwise(expr("timestampdiff(MINUTE, HdrStartTimeGeneratedTmstmp, EventEndTmstmp)")))
                    )

df_Account = spark.read.table('Promotion.dim_Account')\
                        .select("ShipToAccountUUID","ShipToAccountId","ShipToAccountName","PromotionUUID","EffectiveDate","TerminationDate","PerPatientPricingFlag")

df_InvoiceAddendumDetails = spark.read.table("promotion.fact_invoiceaddendumdetails")

df_InvoiceCycleMonth = spark.read.table('Promotion.dim_invoicecyclemonth')\
                                .select("CalendarStartTimeStamp","CalendarEndTimeStamp","InvoiceDate")

df_PatientClassification = spark.read.table('Promotion.dim_patientclassification')\
                                     .select("PatientClassificationUUID","PromotionUUID","PatientClassificationName","DisplayName","ListPrice","EffectiveDate","TerminationDate")

df_SmartCard = spark.read.table('Promotion.dim_smartcard')
df_ConsumerSubscription = spark.read.table('Promotion.fact_consumersubscription')
df_DIMPromotion = spark.read.table('Promotion.DIM_Promotion').select('PromotionUUID','PromotionName','DeviceTypeCd',"EffectiveDate","TerminationDate","PromotionRuleType")

#Used Rank Condition to avoid duplicate records
df_AccountRank = spark.read.table('Promotion.dim_Account')\
                .withColumn("rank",row_number().over(Window.partitionBy("ShipToAccountId","EffectiveDate").orderBy(col("TerminationDate").desc())))\
                .filter("rank == 1")\
                .drop("rank")


df_ConsumerSubscription_ShipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"], "INNER")\
                                .join(df_Account, ["ShipToAccountUUID","PromotionUUID"], "INNER")\
                                .select(df_Account.ShipToAccountId,
                                        df_ConsumerSubscription.CoolSculptingID,
                                        df_ConsumerSubscription.SoldToAccountID,
                                        df_ConsumerSubscription.VersionID,
                                        df_ConsumerSubscription.SubscriptionStartDate,
                                        df_ConsumerSubscription.SubscriptionEndDate,
                                        df_ConsumerSubscription.PilotSubscriptionFlag,
                                        df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                        df_ConsumerSubscription.UpdatedBy,
                                        df_ConsumerSubscription.PromotionUUID
                                    )

#AccountFlag Details                        
df_accountflag = spark.read.table("promotion.fact_accountflag")
df_flagtype = spark.read.table("promotion.dim_flagtype").filter(col("Active") == 1)
df_accountflagtype = (df_accountflag.join(df_flagtype,(df_accountflag.FlagTypeUUID == df_flagtype.FlagTypeUUID),"inner")
                                .join(df_Account,(df_accountflag.ShipToAccountUUID == df_Account.ShipToAccountUUID) & (df_accountflag.PromotionUUID == df_Account.PromotionUUID),"inner")
                                .filter("FlagType != 'HighPerformer'")
                                .select(df_Account.ShipToAccountId,
                                        df_Account.EffectiveDate.alias("Account_EffectiveDate"),
                                        df_Account.TerminationDate.alias("Account_TerminationDate"),
                                        df_accountflag.PromotionUUID,
                                        df_flagtype.FlagType,
                                        df_flagtype.FlagRank,
                                        df_accountflag.FlagStartDate,
                                        df_accountflag.FlagEndDate))    


df_applicator = spark.read.table("silverzone.DIMApplicatorVerCd")                          

In [0]:
def getValidCyclesOverride(InvoiceDate):
    df_dimcyclesoverride = spark.read.table("promotion.dim_cyclesoverride").filter("WorkflowStatus == 'Success'")

    df_cyclesoverride = (df_dimcyclesoverride.join(df_InvoiceCycleMonth,(df_dimcyclesoverride.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp) & (df_InvoiceCycleMonth.InvoiceDate == f'{InvoiceDate}')),"inner")
                .join(df_ULLogs,(df_dimcyclesoverride.CycleId == df_ULLogs.CycleID),"inner")
                .drop(df_dimcyclesoverride.CycleId))

    df_cyclesoverridefnl = (df_cyclesoverride.join(df_DIMPromotion,(df_cyclesoverride.PromotionUUID == df_DIMPromotion.PromotionUUID),"inner")
                                            .join(df_AccountRank, (df_cyclesoverride.ShipToAccountID == df_AccountRank.ShipToAccountId) & (df_cyclesoverride.PromotionUUID == df_AccountRank.PromotionUUID), 'inner')
                                            .join(df_SmartCard, (df_cyclesoverride.CardSPSerialNbr == df_SmartCard.SmartCardSerialNumber), 'left')
                                            .join(df_applicator,(substring(df_cyclesoverride.ApplicatorConfigurantionNbr,1,7) == df_applicator.SSAPPIdentID),"left")\
                                            .filter((df_cyclesoverride.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate)) 
                                            &
                                            ((df_cyclesoverride.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd) | (df_cyclesoverride.DeviceTypeOverrideFlag == True))
                                            &
                                            ((df_AccountRank.PerPatientPricingFlag == 1) | (df_cyclesoverride.AccountP3FlagOverrideFlag == True) | (df_cyclesoverride.HdrStartDateGeneratedDt.between(df_AccountRank.EffectiveDate, df_AccountRank.TerminationDate)))
                                            &
                                            ((df_cyclesoverride.HdrStartDateGeneratedDt >= df_AccountRank.EffectiveDate) | (df_cyclesoverride.AccountEffectiveDateOverrideFlag == True))
                                            &
                                            ((df_cyclesoverride.HdrStartDateGeneratedDt <= df_AccountRank.TerminationDate) | (df_cyclesoverride.AccountTerminationDateOverrideFlag == True))
                                            &
                                            ((df_SmartCard.SmartCardSerialNumber.isNotNull()) | (df_cyclesoverride.SmartCardOverrideFlag == True))
                                            &
                                            ((df_cyclesoverride.HdrStartDateGeneratedDt >= df_SmartCard.EffectiveDate) | (df_cyclesoverride.SmartCardEffectiveDateOverrideFlag == True) | (df_cyclesoverride.SmartCardOverrideFlag == True))
                                            &
                                            ((df_cyclesoverride.HdrStartDateGeneratedDt <= df_SmartCard.TerminationDate) | (df_cyclesoverride.SmartCardTerminationDateOverrideFlag == True) | (df_cyclesoverride.SmartCardOverrideFlag == True))
                                            &
                                            (df_cyclesoverride.PlanType.isin('SmallTreatmentPlan','ComprehensiveTreatmentPlan')))
                                            .withColumn("IsValidEfficacy",when(df_ULLogs.TreatmentTime >= df_applicator.FullEfficacy,lit(1)).otherwise(lit(0)))
                                            .select(df_cyclesoverride.ShipToAccountID,
                                            df_cyclesoverride.CycleID,
                                            df_cyclesoverride.SoldToAccountID,
                                            df_cyclesoverride.CoolSculptingID,
                                            df_cyclesoverride.PromotionUUID,
                                            df_cyclesoverride.HdrStartDateGeneratedDt,
                                            df_cyclesoverride.HdrStartTimeGeneratedTmstmp,
                                            df_cyclesoverride.CreatedDt,
                                            df_cyclesoverride.InvoiceDate,
                                            df_cyclesoverride.CalendarEndTimeStamp,
                                            df_cyclesoverride.CalendarStartTimeStamp,
                                            df_DIMPromotion.PromotionRuleType,
                                            df_cyclesoverride.PlanType,
                                            "IsValidEfficacy")
                                            .distinct())

    return df_cyclesoverridefnl

In [0]:
#Valid P3 Eligible Cycles
def validP3EligibleCycles(InvoiceDate,RuleType=None):  

    #P3 Eligible Cycles
    df_P3Cycles = df_ULLogs.join(broadcast(df_DIMPromotion),(df_ULLogs.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd),"INNER")\
                            .join(df_AccountRank,(df_AccountRank.ShipToAccountId == df_ULLogs.ShipToAccountID) & (df_AccountRank.PromotionUUID == df_DIMPromotion.PromotionUUID),"INNER")\
                            .join(df_SmartCard,(df_ULLogs.CardSPSerialNbr == df_SmartCard.SmartCardSerialNumber) & (df_DIMPromotion.PromotionUUID == df_SmartCard.PromotionUUID),"INNER")\
                            .join(df_InvoiceCycleMonth,(df_ULLogs.CreatedDt.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"INNER")\
                            .join(df_applicator,(substring(df_ULLogs.ApplicatorConfigurantionNbr,1,7) == df_applicator.SSAPPIdentID),"LEFT")\
                            .filter((df_ULLogs.HdrStartDateGeneratedDt.between(df_AccountRank.EffectiveDate, df_AccountRank.TerminationDate)) & (df_ULLogs.HdrStartDateGeneratedDt.between(df_SmartCard.EffectiveDate, df_SmartCard.TerminationDate)) & (df_ULLogs.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate)) & (df_ULLogs.PlanType.isin('SmallTreatmentPlan','ComprehensiveTreatmentPlan')))\
                            .withColumn("IsValidEfficacy",when(df_ULLogs.TreatmentTime >= df_applicator.FullEfficacy,lit(1)).otherwise(lit(0)))\
                            .select(df_ULLogs.ShipToAccountID,
                                    df_ULLogs.CycleID,
                                    df_ULLogs.SoldToAccountID,
                                    df_ULLogs.CoolSculptingID,
                                    df_AccountRank.PromotionUUID,
                                    df_ULLogs.HdrStartDateGeneratedDt,
                                    df_ULLogs.HdrStartTimeGeneratedTmstmp,
                                    df_ULLogs.CreatedDt,
                                    df_InvoiceCycleMonth.InvoiceDate,
                                    df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                    df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                    df_DIMPromotion.PromotionRuleType,
                                    df_ULLogs.PlanType,
                                    "IsValidEfficacy")

    df_cyclesoverridefnl = getValidCyclesOverride(InvoiceDate)
    df_cycles = df_cyclesoverridefnl.unionByName(df_P3Cycles).distinct()

    windowspec = Window.partitionBy("CycleID").orderBy(asc("FlagRank"))

    if RuleType == 'CycleCount':
        df_ULSource = (df_cycles.alias("cycles").join(df_accountflagtype.alias("accountflag"),(df_cycles.ShipToAccountID == df_accountflagtype.ShipToAccountId) & (df_cycles.HdrStartDateGeneratedDt.between(df_accountflagtype.FlagStartDate,df_accountflagtype.FlagEndDate)) & (df_cycles.PromotionUUID == df_accountflagtype.PromotionUUID) & (df_cycles.HdrStartDateGeneratedDt.between(df_accountflagtype.Account_EffectiveDate,df_accountflagtype.Account_TerminationDate)),"left")
                        .withColumn("rank",row_number().over(windowspec))
                        .filter("rank == 1")
                        .groupBy(col("cycles.ShipToAccountID").alias("ShipToAccountId"),
                                col("cycles.SoldToAccountID"),
                                col("cycles.CoolSculptingID"),
                                col("cycles.PromotionUUID"),
                                col("cycles.HdrStartDateGeneratedDt"),
                                col("cycles.CalendarEndTimeStamp"),
                                col("cycles.CalendarStartTimeStamp"),
                                col("accountflag.FlagType"),
                                col("cycles.PromotionRuleType"))
                        .agg(max(col("cycles.CreatedDt")).alias("UL_CreatedDt"),
                        min(col("cycles.HdrStartTimeGeneratedTmstmp")).alias("HdrStartTimeGeneratedTmstmp"),
                        count("*").alias("TotalCycleCount")))
    else:
        #To Identify on the same day, if we received both valid and Invalid efficacy
        windowspecEfficacy = Window.partitionBy("ShipToAccountId","SoldToAccountID","CoolSculptingID","PromotionUUID","HdrStartDateGeneratedDt").orderBy(asc("HdrStartDateGeneratedDt"))
        #To Identify on the same day, if we received both Small and Comprehensive Treatment Plan 
        windowspecPlanType = Window.partitionBy("ShipToAccountId","SoldToAccountID","CoolSculptingID","PromotionUUID","HdrStartDateGeneratedDt").orderBy(asc("HdrStartDateGeneratedDt"))
        df_ULSource = (df_cycles.alias("cycles").join(df_accountflagtype.alias("accountflag"),(df_cycles.ShipToAccountID == df_accountflagtype.ShipToAccountId) & (df_cycles.HdrStartDateGeneratedDt.between(df_accountflagtype.FlagStartDate,df_accountflagtype.FlagEndDate)) & (df_cycles.PromotionUUID == df_accountflagtype.PromotionUUID) & (df_cycles.HdrStartDateGeneratedDt.between(df_accountflagtype.Account_EffectiveDate,df_accountflagtype.Account_TerminationDate)),"left")
                .withColumn("rank",row_number().over(windowspec))
                .withColumn("IsValidEfficacy",when(col("cycles.PromotionRuleType") == 'Consumer',col("cycles.IsValidEfficacy")).otherwise(lit(1)))
                .withColumn("PlanType",when(col("cycles.CoolSculptingID") == 'MISSING','ComprehensiveTreatmentPlan').otherwise(col("cycles.PlanType")))
                .filter("rank == 1")
                .groupBy(col("cycles.ShipToAccountID").alias("ShipToAccountId"),
                        col("cycles.SoldToAccountID"),
                        col("cycles.CoolSculptingID"),
                        col("cycles.PromotionUUID"),
                        col("cycles.HdrStartDateGeneratedDt"),
                        col("cycles.CalendarEndTimeStamp"),
                        col("cycles.CalendarStartTimeStamp"),
                        col("accountflag.FlagType"),
                        col("cycles.PromotionRuleType"),
                        col("PlanType"),
                        col("IsValidEfficacy"))
                .agg(max(col("cycles.CreatedDt")).alias("UL_CreatedDt"),
                min(col("cycles.HdrStartTimeGeneratedTmstmp")).alias("HdrStartTimeGeneratedTmstmp"),
                count("*").alias("TotalCycleCount"))
                .withColumn("IsMultipleEfficacy",size(collect_set(col("IsValidEfficacy")).over(windowspecEfficacy)))
                .withColumn("IsMultiplePlanType",size(collect_set(col("PlanType")).over(windowspecPlanType)))
        )
            
    return df_ULSource

In [0]:
def previousInvoicedRecord():
    df_DIMPromotion = spark.read.table('Promotion.DIM_Promotion').filter(f'"{InvoiceDate}" >= EffectiveDate and "{CalendarStartDate}" <= TerminationDate').select('PromotionUUID','PromotionName','DeviceTypeCd')
    
    df_invoiceaddendum_invoiceaddendumdetails = spark.read.table("promotion.fact_invoiceaddendum_invoiceaddendumdetails")\
                                            .filter('IsSnapShotFlag == 1')\
                                            .join(df_InvoiceAddendumDetails,["InvoiceAddendumDetailsUUID"],"INNER")\
                                            .join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                            .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                            .select(df_Account.ShipToAccountId,
                                                    df_InvoiceAddendumDetails.CycleDate,
                                                    df_InvoiceAddendumDetails.SoldToAccountID,
                                                    df_DIMPromotion.PromotionUUID,
                                                    df_PatientClassification.PatientClassificationName,
                                                    df_InvoiceAddendumDetails.CoolSculptingID,
                                                    df_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID.alias("PreviousInvoiceAddendumDetailsUUID"),
                                                    lit(1).alias("IsInvoiced"))\
                                            .filter(~col("PatientClassificationName").isin('NoPatientAssociationFeeExceptionPostInvoice','NoPatientAssociationFeeExceptionPreInvoice','NoPatientAssociationFeePerCycleException','PerPatientFeeExceptionPostInvoice','PerPatientFeeExceptionPreInvoice','PerPatientFeeException','Exception'))
    return df_invoiceaddendum_invoiceaddendumdetails

In [0]:
def currentInvoiceRecord(InvoiceDate,RuleType=None):
    df_DIMPromotion = spark.read.table('Promotion.DIM_Promotion').filter(f'"{InvoiceDate}" >= EffectiveDate and "{CalendarStartDate}" <= TerminationDate').select('PromotionUUID','PromotionName','DeviceTypeCd')

    df_InvoiceCycleMonth = spark.read.table('Promotion.dim_invoicecyclemonth')\
                                .select("CalendarStartTimeStamp","CalendarEndTimeStamp","InvoiceDate")
    if RuleType == "CycleCount":
        df_Invoicetarget = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"INNER")\
                                .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                .join(df_ConsumerSubscription,["ConsumerSubscriptionUUID","PromotionUUID"],"LEFT")\
                                .filter(~col("PatientClassificationName").isin('NoPatientAssociationFeeExceptionPostInvoice','NoPatientAssociationFeeExceptionPreInvoice','NoPatientAssociationFeePerCycleException','PerPatientFeeExceptionPostInvoice','PerPatientFeeExceptionPreInvoice','PerPatientFeeException','Exception','PerCycleFeeExceptionPostInvoice','PerCycleFeeExceptionPreInvoice'))\
                                .filter(df_InvoiceAddendumDetails.VersionID == 1)\
                                .select(df_Account.ShipToAccountId,
                                        df_InvoiceAddendumDetails.CoolSculptingID,
                                        df_InvoiceAddendumDetails.SoldToAccountID,
                                        df_PatientClassification.PatientClassificationName,
                                        df_InvoiceAddendumDetails.CycleDate,
                                        df_InvoiceAddendumDetails.CycleCount,
                                        df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                        df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                        df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                        df_InvoiceAddendumDetails.CreatedBy,
                                        df_InvoiceAddendumDetails.CreatedDate,
                                        df_InvoiceAddendumDetails.UpdatedDate,
                                        df_ConsumerSubscription.PilotSubscriptionFlag,
                                        df_InvoiceAddendumDetails.PromotionUUID
                                        )\
                                .groupBy("ShipToAccountId","SoldToAccountID","CoolSculptingID","CycleDate","CalendarStartTimeStamp","CalendarEndTimeStamp","PilotSubscriptionFlag","CreatedBy","PromotionUUID")\
                                .agg(sum("CycleCount").alias("CycleCount"),
                                sum("IncrementalInvoiceCharge").alias("IncrementalInvoiceCharge"),
                                max(df_InvoiceAddendumDetails.CreatedDate).alias("Details_CreatedDt"),
                                max(df_InvoiceAddendumDetails.UpdatedDate).alias("Details_UpdatedDt")
                                )
    else:
        df_Invoicetarget = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"INNER")\
                                .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                .join(df_ConsumerSubscription,["ConsumerSubscriptionUUID","PromotionUUID"],"LEFT")\
                                .filter(~col("PatientClassificationName").isin('NoPatientAssociationFeeExceptionPostInvoice','NoPatientAssociationFeeExceptionPreInvoice','NoPatientAssociationFeePerCycleException','PerPatientFeeExceptionPostInvoice','PerPatientFeeExceptionPreInvoice','PerPatientFeeException','Exception','PerCycleFeeExceptionPostInvoice','PerCycleFeeExceptionPreInvoice'))\
                                .filter(df_InvoiceAddendumDetails.VersionID == 1)\
                                .select(df_Account.ShipToAccountId,
                                        df_InvoiceAddendumDetails.CoolSculptingID,
                                        df_InvoiceAddendumDetails.SoldToAccountID,
                                        df_PatientClassification.PatientClassificationName,
                                        df_InvoiceAddendumDetails.CycleDate,
                                        df_InvoiceAddendumDetails.CycleCount,
                                        df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                        df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                        df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                        df_InvoiceAddendumDetails.CreatedBy,
                                        df_InvoiceAddendumDetails.CreatedDate,
                                        df_InvoiceAddendumDetails.UpdatedDate,
                                        df_ConsumerSubscription.PilotSubscriptionFlag,
                                        df_InvoiceAddendumDetails.PromotionUUID
                                        )\
                                .groupBy("ShipToAccountId","SoldToAccountID","CoolSculptingID","PatientClassificationName","CycleDate","CalendarStartTimeStamp","CalendarEndTimeStamp","PilotSubscriptionFlag","CreatedBy","PromotionUUID")\
                                .agg(sum("CycleCount").alias("CycleCount"),
                                sum("IncrementalInvoiceCharge").alias("IncrementalInvoiceCharge"),
                                max(df_InvoiceAddendumDetails.CreatedDate).alias("Details_CreatedDt"),
                                max(df_InvoiceAddendumDetails.UpdatedDate).alias("Details_UpdatedDt")
                                )
    return df_Invoicetarget

In [0]:
def fetchInvoiceExceptionRecord(InvoiceDate):
    #Invoice Exception Comments Validation
    df_InvoiceException = (spark.read.table("Promotion.fact_invoiceexception")
            .withColumn('CoolSculptingID',expr('CASE WHEN PhoneNumber IS NULL OR PhoneNumber IN ("UNKNOWN","MISSING","","0000000000") THEN "MISSING" ELSE PhoneNumber END'))
            .filter((col("IsProcessedFlag") == True) & (col("Comments") == 'Completed'))
            .join(df_InvoiceCycleMonth,(col("CreatedDate").between(col("CalendarStartTimeStamp"), col("CalendarEndTimeStamp"))) & (col("InvoiceDate") == lit(f'{InvoiceDate}')),"INNER"))

    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    df_ConsumerSubscription_shipto = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"], "INNER")\
                                                            .join(df_Account, ["ShipToAccountUUID","PromotionUUID"], "INNER")\
                                                            .select(df_Account.ShipToAccountId,
                                                                df_ConsumerSubscription.CoolSculptingID,
                                                                df_ConsumerSubscription.SoldToAccountID,
                                                                df_ConsumerSubscription.VersionID,
                                                                df_ConsumerSubscription.SubscriptionStartDate,
                                                                df_ConsumerSubscription.SubscriptionEndDate,
                                                                df_ConsumerSubscription.PilotSubscriptionFlag,
                                                                df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                                                df_ConsumerSubscription.UpdatedBy,
                                                                df_ConsumerSubscription.PromotionUUID,
                                                                df_ConsumerSubscription.CreatedDate
                                                             )\
                                                             .withColumn("PreviousSubscriptionEndDate",lag("SubscriptionEndDate").over(Window.partitionBy("CoolSculptingID","ShipToAccountId").orderBy(asc("CreatedDate"))))\
                                                             .filter("VersionID ==1")

    df_InvoiceExceptionDetails = df_InvoiceException.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                    .join(df_invoiceaddendum_invoiceaddendumdetails,["CoolSculptingID","ShipToAccountId","CycleDate"],"LEFT")\
                                    .join(df_ConsumerSubscription_shipto,(df_InvoiceException.ShipToAccountID == df_ConsumerSubscription_shipto.ShipToAccountId) & (df_InvoiceException.CoolSculptingID == df_ConsumerSubscription_shipto.CoolSculptingID) & (df_InvoiceException.CycleDate.between(df_ConsumerSubscription_shipto.SubscriptionStartDate,df_ConsumerSubscription_shipto.SubscriptionEndDate)),"LEFT")\
                                    .select(df_InvoiceException.ShipToAccountID,
                                            df_InvoiceException.CoolSculptingID,
                                            df_InvoiceException.PromotionUUID,
                                            df_InvoiceException.MoxieCaseNumber,
                                            df_InvoiceException.IncrementalInvoiceCharge,
                                            df_InvoiceException.ConsumerSubscriptionEndDate,
                                            df_InvoiceException.CycleCount,
                                            df_InvoiceException.CycleDate,
                                            df_InvoiceException.CreatedDate.alias("Exception_CreatedDt"),
                                            df_InvoiceException.UpdatedDate.alias("Exception_UpdatedDt"),
                                            coalesce(df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced,lit('0')).alias("IsInvoiced"),
                                            df_ConsumerSubscription_shipto.SoldToAccountID,
                                            df_ConsumerSubscription_shipto.SubscriptionStartDate,
                                            df_ConsumerSubscription_shipto.SubscriptionEndDate,
                                            df_ConsumerSubscription_shipto.PreviousSubscriptionEndDate,
                                            df_DIMPromotion.PromotionRuleType)\
                                    .withColumn("CommentsDerived",when((col("IsInvoiced") == '1') & (col("IncrementalInvoiceCharge") == "950") & (col("ConsumerSubscriptionEndDate").isNull()),"Exception_PerPatientPostInvoice")
                                            .when((col("IsInvoiced") == '0') & (col("IncrementalInvoiceCharge") == "950") & (col("ConsumerSubscriptionEndDate").isNull()),"Exception_PerPatientPreInvoice")
                                            .when((col("IsInvoiced") == '1') & (col("IncrementalInvoiceCharge") != "950") & (col("IncrementalInvoiceCharge")/col("CycleCount") == 260),"Exception_NoPatientPostInvoice")
                                            .when((col("IsInvoiced") == '0') & (col("IncrementalInvoiceCharge") != "950") & (col("IncrementalInvoiceCharge")/col("CycleCount") == 260),"Exception_NoPatientPreInvoice")
                                            .when((col("IsInvoiced") == '0') & (col("IncrementalInvoiceCharge") != "950") & (col("IncrementalInvoiceCharge")/col("CycleCount") == 130),"Exception_NoPatientPerCycle")
                                            .when((col("ConsumerSubscriptionEndDate").isNotNull()) & ((col("ConsumerSubscriptionEndDate") == col("SubscriptionEndDate")) | (col("ConsumerSubscriptionEndDate") == col("PreviousSubscriptionEndDate"))),"Exception_SubscriptionDateChange")
                                            .when((col("IsInvoiced") == '1') & (col("IncrementalInvoiceCharge") != "950") & (col("IncrementalInvoiceCharge")/col("CycleCount") == 200),"Exception_PerCycleFeePostInvoice")
                                            .when((col("IsInvoiced") == '0') & (col("IncrementalInvoiceCharge") != "950") & (col("IncrementalInvoiceCharge")/col("CycleCount") == 200),"Exception_PerCycleFeePreInvoice"))\
                                    .withColumn("PatientClassificationNameDerived",when((col("CommentsDerived") == 'Exception_PerPatientPostInvoice'),"PerPatientFeeExceptionPostInvoice")
                                                                                .when((col("CommentsDerived") == 'Exception_PerPatientPreInvoice'),"PerPatientFeeExceptionPreInvoice")
                                                                                .when((col("CommentsDerived") == 'Exception_NoPatientPostInvoice'),"NoPatientAssociationFeeExceptionPostInvoice")
                                                                                .when((col("CommentsDerived") == 'Exception_NoPatientPreInvoice'),"NoPatientAssociationFeeExceptionPreInvoice")
                                                                                .when((col("CommentsDerived") == 'Exception_NoPatientPerCycle'),"NoPatientAssociationFeePerCycleException")
                                                                                .when((col("CommentsDerived") == 'Exception_PerCycleFeePostInvoice'),"PerCycleFeeExceptionPostInvoice")
                                                                                .when((col("CommentsDerived") == 'Exception_PerCycleFeePreInvoice'),"PerCycleFeeExceptionPreInvoice"))

    df_PerCycleException = df_InvoiceExceptionDetails.filter(col("CommentsDerived") == 'Exception_NoPatientPerCycle')\
                                                    .withColumn("CommentsDerived",lit("Exception_NoPatientPreInvoice"))\
                                                    .withColumn("PatientClassificationNameDerived",lit("NoPatientAssociationFeeExceptionPreInvoice"))\

    df_InvoiceExceptionDetails = df_InvoiceExceptionDetails.unionByName(df_PerCycleException,True)
    return df_InvoiceExceptionDetails

In [0]:
# DifferentShipToID FollowUp/LateCycle/UpdateCycle/Return Subscription
def deriveCommentsConsumer(InvoiceDate):
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Valid P3 Cycle
    df_ullogs = validP3EligibleCycles(InvoiceDate)

    df_source = df_ullogs.filter(col("PromotionRuleType") == 'Consumer')

    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select(df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    df_ConsumerSubscription.SoldToAccountID,
                                                                    df_Account.ShipToAccountId.alias("Subscription_ShipToAccountId"),
                                                                    df_ConsumerSubscription.CoolSculptingID,
                                                                    df_ConsumerSubscription.SubscriptionStartDate,
                                                                    df_ConsumerSubscription.SubscriptionEndDate,
                                                                    df_ConsumerSubscription.PilotSubscriptionFlag,
                                                                    df_ConsumerSubscription.CreatedDate)

    #Find the Previous Subscription Details to check for the return subscription
    df_consumersubscription_previous = df_ConsumerSubscription_shipTo.withColumnRenamed("Subscription_ShipToAccountId","ShipToAccountId")\
                                                                     .withColumnRenamed("SubscriptionStartDate","ConsumerSubscriptionStartDate")\
                                                                     .withColumnRenamed("SubscriptionEndDate","ConsumerSubscriptionEndDate")\
                                                                     .withColumnRenamed("ConsumerSubscriptionUUID","ConsumerSubscriptionUUID_1")\
                                                                .withColumn("PreviousSubscriptionUUID",lag("ConsumerSubscriptionUUID_1").over(Window.partitionBy("CoolSculptingID").orderBy(asc("ConsumerSubscriptionEndDate"),asc("CreatedDate"))))\
                                                                .withColumn("PreviousPilotSubscriptionFlag",lag("PilotSubscriptionFlag").over(Window.partitionBy("CoolSculptingID").orderBy(asc("ConsumerSubscriptionEndDate"))))\
                                                                 .withColumn("IsPreviousSubscription",when(col("PreviousSubscriptionUUID").isNotNull(),lit(1)).otherwise(lit(0)))

    #Join Ullogs with Consumersubscription to find DifferentShipTo records 
    #Join Ullogs with PreviousConsumerSubscription to find Pre Patient Fee and Return Subscription
    #Join Ullogs with Already Invoiced Records to find the Update/Late Cycle
    df_source_classification = df_source.alias("ULLogs").join(df_consumersubscription_previous.alias("PreviousSubscription"),((df_source.CoolSculptingID == df_consumersubscription_previous.CoolSculptingID) & (df_source.HdrStartDateGeneratedDt.between(df_consumersubscription_previous.ConsumerSubscriptionStartDate,df_consumersubscription_previous.ConsumerSubscriptionEndDate)) & (df_source.PromotionUUID == df_consumersubscription_previous.PromotionUUID)),"LEFT")\
                                    .join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(df_source.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_source.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_source.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_source.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                    .withColumn("isDiffernetShipToID",lit(None))\
                                    .withColumn("IsPerPatientFee",when((df_source.HdrStartDateGeneratedDt == df_consumersubscription_previous.ConsumerSubscriptionStartDate) & (df_source.ShipToAccountId == df_consumersubscription_previous.ShipToAccountId),lit(1)))\
                                    .withColumn("IsReturnSubscription",when((col("IsPreviousSubscription") == 1) & (col("IsPerPatientFee") == 1),lit(1)))\
                                    .withColumn("IsUpdateCycle",when((col("IsInvoiced") == 1) & (col("HdrStartDateGeneratedDt") < to_date(col("CalendarStartTimeStamp"))),1))\
                                    .withColumn("IsLateCycle",when((col("IsInvoiced").isNull()) & (col("HdrStartDateGeneratedDt") < to_date(col("CalendarStartTimeStamp"))),1))\
                                    .select(col("ULLogs.ShipToAccountId"),
                                        col("ULLogs.SoldToAccountID"),
                                        col("ULLogs.CoolSculptingID"),
                                        col("ULLogs.PromotionUUID"),
                                        col("ULLogs.HdrStartDateGeneratedDt"),
                                        col("ULLogs.TotalCycleCount"),
                                        col("ULLogs.FlagType"),
                                        col("ULLogs.UL_CreatedDt"),
                                        col("ULLogs.CalendarStartTimeStamp"),
                                        col("ULLogs.CalendarEndTimeStamp"),
                                        col("PreviousSubscription.ShipToAccountId").alias("Subscription_ShipToAccountId"),
                                        col("PreviousSubscription.ConsumerSubscriptionUUID_1").alias("ConsumerSubscriptionUUID"),
                                        col("PreviousSubscription.ConsumerSubscriptionStartDate").alias("SubscriptionStartDate"),
                                        col("PreviousSubscription.ConsumerSubscriptionEndDate").alias("SubscriptionEndDate"),
                                        col("Invoiced.IsInvoiced"),
                                        col("PreviousSubscription.PreviousPilotSubscriptionFlag"),
                                        col("PreviousSubscription.ConsumerSubscriptionUUID_1"),
                                        "isDiffernetShipToID",
                                        "IsPerPatientFee",
                                        "IsReturnSubscription",
                                        "IsUpdateCycle",
                                        "IsLateCycle",
                                        col("ULLogs.PlanType"),
                                        col("ULLogs.IsValidEfficacy"),
                                        col("ULLogs.PromotionRuleType"),
                                        col("ULLogs.IsMultipleEfficacy"),
                                        col("ULLogs.IsMultiplePlanType")).distinct()
                                    
    # If multiple subscriptions are there for a coolsculptingid, pick the subscription that was having furthest Subscription end date
    windowSpec  = Window.partitionBy("CoolSculptingID","PromotionUUID","ShipToAccountId","HdrStartDateGeneratedDt","SoldToAccountID","PlanType","IsValidEfficacy").orderBy(desc("SubscriptionEndDate"))
    df_source_classification = df_source_classification.withColumn("IsMultiSubscripition",row_number().over(windowSpec)).filter(col("IsMultiSubscripition") == 1).drop("IsMultiSubscripition")
    
    return df_source_classification

In [0]:
# DifferentShipToID FollowUp/LateCycle/UpdateCycle/Return Subscription
def deriveCommentsConsumerShipToSoldTo(InvoiceDate):
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Valid P3 Cycle
    df_ullogs = validP3EligibleCycles(InvoiceDate)

    df_source = df_ullogs.filter(col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo')

    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select(df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    df_ConsumerSubscription.SoldToAccountID,
                                                                    df_Account.ShipToAccountId.alias("Subscription_ShipToAccountId"),
                                                                    df_ConsumerSubscription.CoolSculptingID,
                                                                    df_ConsumerSubscription.SubscriptionStartDate,
                                                                    df_ConsumerSubscription.SubscriptionEndDate,
                                                                    df_ConsumerSubscription.PilotSubscriptionFlag,
                                                                    df_ConsumerSubscription.CreatedDate)

    #Find the Previous Subscription Details to check for the return subscription
    df_consumersubscription_previous = df_ConsumerSubscription_shipTo.withColumnRenamed("Subscription_ShipToAccountId","ShipToAccountId")\
                                                                     .withColumnRenamed("SubscriptionStartDate","ConsumerSubscriptionStartDate")\
                                                                     .withColumnRenamed("SubscriptionEndDate","ConsumerSubscriptionEndDate")\
                                                                     .withColumnRenamed("ConsumerSubscriptionUUID","ConsumerSubscriptionUUID_1")\
                                                                .withColumn("PreviousSubscriptionUUID",lag("ConsumerSubscriptionUUID_1").over(Window.partitionBy("CoolSculptingID").orderBy(asc("ConsumerSubscriptionEndDate"),asc("CreatedDate"))))\
                                                                .withColumn("PreviousPilotSubscriptionFlag",lag("PilotSubscriptionFlag").over(Window.partitionBy("CoolSculptingID").orderBy(asc("ConsumerSubscriptionEndDate"))))\
                                                                 .withColumn("IsPreviousSubscription",when(col("PreviousSubscriptionUUID").isNotNull(),lit(1)).otherwise(lit(0)))

    #Join Ullogs with Consumersubscription to find DifferentShipTo records 
    #Join Ullogs with PreviousConsumerSubscription to find Pre Patient Fee and Return Subscription
    #Join Ullogs with Already Invoiced Records to find the Update/Late Cycle
    df_source_classification = df_source.alias("ULLogs").join(df_ConsumerSubscription_shipTo.alias("Subscription"),((df_source.ShipToAccountId != df_ConsumerSubscription_shipTo.Subscription_ShipToAccountId ) & (df_source.SoldToAccountID == df_ConsumerSubscription_shipTo.SoldToAccountID) & (df_source.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID)& (df_source.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate)) & (df_source.PromotionUUID == df_ConsumerSubscription_shipTo.PromotionUUID)),"LEFT")\
                                    .join(df_consumersubscription_previous.alias("PreviousSubscription"),((df_source.ShipToAccountId == df_consumersubscription_previous.ShipToAccountId) & (df_source.CoolSculptingID == df_consumersubscription_previous.CoolSculptingID) & (df_source.HdrStartDateGeneratedDt.between(df_consumersubscription_previous.ConsumerSubscriptionStartDate,df_consumersubscription_previous.ConsumerSubscriptionEndDate)) & (df_source.PromotionUUID == df_consumersubscription_previous.PromotionUUID)),"LEFT")\
                                    .join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(df_source.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_source.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_source.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_source.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                    .withColumn("isDiffernetShipToID",when(df_ConsumerSubscription_shipTo.ConsumerSubscriptionUUID.isNotNull(),lit(1)))\
                                    .withColumn("IsPerPatientFee",when(df_source.HdrStartDateGeneratedDt == df_consumersubscription_previous.ConsumerSubscriptionStartDate,lit(1)))\
                                    .withColumn("IsReturnSubscription",when((col("IsPreviousSubscription") == 1) & (col("IsPerPatientFee") == 1),lit(1)))\
                                    .withColumn("IsUpdateCycle",when((col("IsInvoiced") == 1) & (col("HdrStartDateGeneratedDt") < to_date(col("CalendarStartTimeStamp"))),1))\
                                    .withColumn("IsLateCycle",when((col("IsInvoiced").isNull()) & (col("HdrStartDateGeneratedDt") < to_date(col("CalendarStartTimeStamp"))),1))\
                                    .select(col("ULLogs.ShipToAccountId"),
                                        col("ULLogs.SoldToAccountID"),
                                        col("ULLogs.CoolSculptingID"),
                                        col("ULLogs.PromotionUUID"),
                                        col("ULLogs.HdrStartDateGeneratedDt"),
                                        col("ULLogs.TotalCycleCount"),
                                        col("ULLogs.FlagType"),
                                        col("ULLogs.UL_CreatedDt"),
                                        col("ULLogs.CalendarStartTimeStamp"),
                                        col("ULLogs.CalendarEndTimeStamp"),
                                        col("Subscription.Subscription_ShipToAccountId"),
                                        col("Subscription.ConsumerSubscriptionUUID"),
                                        col("Subscription.SubscriptionStartDate"),
                                        col("Subscription.SubscriptionEndDate"),
                                        col("Invoiced.IsInvoiced"),
                                        col("PreviousSubscription.PreviousPilotSubscriptionFlag"),
                                        col("PreviousSubscription.ConsumerSubscriptionUUID_1"),
                                        "isDiffernetShipToID",
                                        "IsPerPatientFee",
                                        "IsReturnSubscription",
                                        "IsUpdateCycle",
                                        "IsLateCycle",
                                        col("ULLogs.PlanType"),
                                        col("ULLogs.IsValidEfficacy"),
                                        col("ULLogs.PromotionRuleType"),
                                        col("ULLogs.IsMultipleEfficacy"),
                                        col("ULLogs.IsMultiplePlanType")).distinct()

    return df_source_classification

In [0]:
def subscriptionAdjustedConsumer(InvoiceDate):
    #P3 Eligible Cycles
    df_ULogs = validP3EligibleCycles(InvoiceDate)

    df_ULSource = df_ULogs.filter(col("PromotionRuleType") == 'Consumer')

    #Previous Susbcription details are retrieved
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                        .join(df_Account,["ShipToAccountUUID","PromotionUUID"], "INNER")\
                                        .filter(df_ConsumerSubscription.VersionID == 1)\
                                        .withColumn("LeadConsumerSubscriptionUUID",lead(col("ConsumerSubscriptionUUID")).over(Window.partitionBy(col("CoolSculptingID"),col("PromotionUUID")).orderBy(col("SubscriptionEndDate").asc())))\
                                        .withColumn("LeadSubscriptionStartDate",lead(col("SubscriptionStartDate")).over(Window.partitionBy(col("CoolSculptingID"),col("PromotionUUID")).orderBy(col("SubscriptionEndDate").asc())))\
                                        .select(df_Account.ShipToAccountId,
                                            df_ConsumerSubscription.SoldToAccountID,
                                            df_ConsumerSubscription.CoolSculptingID,
                                            df_ConsumerSubscription.PromotionUUID,
                                            df_ConsumerSubscription.VersionID,
                                            df_ConsumerSubscription.SubscriptionStartDate,
                                            df_ConsumerSubscription.SubscriptionEndDate,
                                            df_ConsumerSubscription.PilotSubscriptionFlag,
                                            df_ConsumerSubscription.InvoiceExceptionFlag,
                                            df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                            df_ConsumerSubscription.PreviousConsumerSubscriptionUUID,
                                            df_ConsumerSubscription.UpdatedBy,
                                            "LeadConsumerSubscriptionUUID",
                                            "LeadSubscriptionStartDate"
                                        )
    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()
    
    # Fetch Previous Version Subscription details to find the Subscription Adjustment.
    df_PreviousConsumerSubscription_2 = spark.read.table('Promotion.fact_consumersubscription')\
                                            .filter(col('VersionID') == 2)\
                                            .select(col("ConsumerSubscriptionUUID").alias("ConsumerSubscriptionUUID_2"),
                                                    col("PreviousConsumerSubscriptionUUID").alias("PreviousConsumerSubscriptionUUID_2"),
                                                    col("SubscriptionStartDate").alias("SubscriptionStartDate_2"),
                                                    col("SubscriptionEndDate").alias("SubscriptionEndDate_2"),
                                                    col("Comments").alias("Comments_2"))
    
    df_ConsumerSubscription_2 = (df_PreviousConsumerSubscription_2.alias("Subscription_2").join(df_PreviousConsumerSubscription_2.alias("PreviousSubscription_2"),(col("Subscription_2.PreviousConsumerSubscriptionUUID_2") == col("PreviousSubscription_2.ConsumerSubscriptionUUID_2")),"LEFT")
                                 .select(col("Subscription_2.ConsumerSubscriptionUUID_2"),
                                         col("Subscription_2.PreviousConsumerSubscriptionUUID_2"),
                                         col("Subscription_2.SubscriptionStartDate_2"),
                                         col("Subscription_2.SubscriptionEndDate_2"),
                                         col("Subscription_2.Comments_2"),
                                         col("PreviousSubscription_2.SubscriptionStartDate_2").alias("PreviousSubscriptionStartDate_2"),
                                         col("PreviousSubscription_2.SubscriptionEndDate_2").alias("PreviousSubscriptionEndDate_2")))
                                            
    # Join Ullogs with Consumer Subscription and join the previous version Subscription  details  
    # If the Treatment Date was lesser than the previous version subscriptionStatDate and equal to the Current SubscriptionStartDate then the record is considered as IsSubscriptionAdjustedFollowUp
    # If new Subscription Started after the Current Subscription then the record is considered as IsSubscriptionReAllignment
    # If Exception was created after TruUp Scenario, check the previous subscription of version 2 and if PreviousSubscriptionDate was greater than the Current SubscriptionStartDate then the record is considered as IsSubscriptionAdjustedFollowUp

    df_InvoicedAdjustedRecord = df_ULSource.alias("ULLogs").join(df_ConsumerSubscription_shipTo.alias("Subscription"),((col("ULLogs.CoolSculptingID") == col("Subscription.CoolSculptingID")) & (col("ULLogs.HdrStartDateGeneratedDt").between(col("Subscription.SubscriptionStartDate"),col("Subscription.SubscriptionEndDate"))) & (col("ULLogs.PromotionUUID") == col("Subscription.PromotionUUID"))),"LEFT")\
                    .join(df_ConsumerSubscription_2.alias("Subscription_2"),(col("Subscription.PreviousConsumerSubscriptionUUID") == col("Subscription_2.ConsumerSubscriptionUUID_2")),"LEFT")\
                    .withColumn("IsSubscriptionAdjustedFollowUp",when(((col("ULLogs.HdrStartDateGeneratedDt") < col("Subscription_2.SubscriptionStartDate_2")) | ((col("Subscription.InvoiceExceptionFlag") == 'Y') & (col("ULLogs.HdrStartDateGeneratedDt") < col("Subscription_2.PreviousSubscriptionStartDate_2")))) & (col("ULLogs.HdrStartDateGeneratedDt") == col("Subscription.SubscriptionStartDate")),lit(1)))\
                    .withColumn("IsSubscriptionReAllignment",when((col("IsSubscriptionAdjustedFollowUp") == 1) & (col("Subscription.LeadConsumerSubscriptionUUID").isNotNull()),lit(1)))\
                    .withColumn("IsSubscriptionAdjustedFee",when((col("ULLogs.HdrStartDateGeneratedDt") == col("Subscription_2.SubscriptionStartDate_2")) | (col("ULLogs.HdrStartDateGeneratedDt") == col("Subscription_2.PreviousSubscriptionStartDate_2")),lit(1)))\
                            .select(col("ULLogs.ShipToAccountId"),
                                            col("ULLogs.SoldToAccountID"),
                                            col("ULLogs.CoolSculptingID"),
                                            col("ULLogs.PromotionUUID"),
                                            col("ULLogs.HdrStartDateGeneratedDt"),
                                            col("ULLogs.CalendarEndTimeStamp"),
                                            col("ULLogs.CalendarStartTimeStamp"),
                                            col("ULLogs.UL_CreatedDt"),
                                            col("ULLogs.TotalCycleCount"),
                                            col("Subscription.SubscriptionStartDate"),
                                            col("Subscription.SubscriptionEndDate"),
                                            col("Subscription.ConsumerSubscriptionUUID"),
                                            col("Subscription.PreviousConsumerSubscriptionUUID"),
                                            col("Subscription_2.ConsumerSubscriptionUUID_2"),
                                            col("Subscription_2.PreviousConsumerSubscriptionUUID_2"),
                                            col("Subscription_2.SubscriptionStartDate_2"),
                                            col("Subscription_2.SubscriptionEndDate_2"),
                                            col("Subscription_2.Comments_2"),
                                            "IsSubscriptionAdjustedFollowUp",
                                            "IsSubscriptionReAllignment",
                                            "IsSubscriptionAdjustedFee",
                                            col("ULLogs.PlanType")
                                            )
 
    # If multiple subscriptions are there for a coolsculptingid, pick the subscription that was having furthest Subscription end date
    windowSpec  = Window.partitionBy("CoolSculptingID","PromotionUUID","ShipToAccountId","HdrStartDateGeneratedDt","SoldToAccountID").orderBy(desc("SubscriptionEndDate"))
    df_InvoicedAdjustedRecord = df_InvoicedAdjustedRecord.withColumn("IsMultiSubscripition",row_number().over(windowSpec)).filter(col("IsMultiSubscripition") == 1)
    
    df_InvoicedAdjustedRecordTemp = df_InvoicedAdjustedRecord.filter("IsSubscriptionAdjustedFollowUp == 1")\
                                                             .withColumnRenamed("IsSubscriptionAdjustedFollowUp","IsSubscriptionAdjustedFollowUp_1")\
                                                             .withColumnRenamed("CoolSculptingID","CoolSculptingID_1")

    #Find the already Invoiced SubscriptionAdjusted records
    df_InvoicedAdjustedRecord_1 = df_InvoicedAdjustedRecordTemp.alias("FollowUp").join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(df_InvoicedAdjustedRecordTemp.CoolSculptingID_1 == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_InvoicedAdjustedRecordTemp.HdrStartDateGeneratedDt < df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_InvoicedAdjustedRecordTemp.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                    .select(col("Invoiced.ShipToAccountId"),
                                            col("Invoiced.CoolSculptingID"),
                                            col("Invoiced.CycleDate").alias("HdrStartDateGeneratedDt"),
                                            col("Invoiced.PatientClassificationName"),
                                            col("Invoiced.PreviousInvoiceAddendumDetailsUUID").alias("InvoiceAddendumDetailsUUID"),
                                            col("Invoiced.IsInvoiced"),
                                            col("Invoiced.PromotionUUID"),
                                            when(col("Invoiced.PreviousInvoiceAddendumDetailsUUID").isNotNull(),lit(1)).alias("IsSubscriptionAdjusted")
                                            )\
                                    .filter(col("Invoiced.ShipToAccountId").isNotNull())

    #Find the current Invoice SubscriptionAdjusted records
    df_InvoicedAdjustedRecord_2 = df_InvoicedAdjustedRecordTemp.alias("src").join(df_InvoicedAdjustedRecord.alias("tgt"),(df_InvoicedAdjustedRecordTemp.CoolSculptingID_1 == df_InvoicedAdjustedRecord.CoolSculptingID) & (df_InvoicedAdjustedRecord.IsSubscriptionAdjustedFollowUp.isNull() & (df_InvoicedAdjustedRecordTemp.PromotionUUID == df_InvoicedAdjustedRecord.PromotionUUID)),"LEFT")\
                                                                .select(col("tgt.ShipToAccountId"),
                                                                        col("tgt.CoolSculptingID"),
                                                                        col("tgt.HdrStartDateGeneratedDt"),
                                                                        col("tgt.PromotionUUID"),
                                                                        when(col("tgt.ShipToAccountId").isNotNull(),lit(1)).alias("IsSubscriptionAdjustedNotInvoiced"))\
                                                                .filter(col("tgt.ShipToAccountId").isNotNull())
    
    #Combine Both the Invoiced/NotInvoiced SubscriptionsAdjusted records
    df_InvoicedAdjustedRecord_3 = df_InvoicedAdjustedRecord_1.join(df_InvoicedAdjustedRecord_2,(df_InvoicedAdjustedRecord_1.ShipToAccountId == df_InvoicedAdjustedRecord_2.ShipToAccountId) & (df_InvoicedAdjustedRecord_1.CoolSculptingID == df_InvoicedAdjustedRecord_2.CoolSculptingID) & (df_InvoicedAdjustedRecord_1.HdrStartDateGeneratedDt == df_InvoicedAdjustedRecord_2.HdrStartDateGeneratedDt) & (df_InvoicedAdjustedRecord_1.PromotionUUID == df_InvoicedAdjustedRecord_2.PromotionUUID),"FULL")\
                                                             .select(coalesce(df_InvoicedAdjustedRecord_1.ShipToAccountId,df_InvoicedAdjustedRecord_2.ShipToAccountId).alias("ShipToAccountId"),
                                                                     coalesce(df_InvoicedAdjustedRecord_1.CoolSculptingID,df_InvoicedAdjustedRecord_2.CoolSculptingID).alias("CoolSculptingID"),
                                                                     coalesce(df_InvoicedAdjustedRecord_1.HdrStartDateGeneratedDt,df_InvoicedAdjustedRecord_2.HdrStartDateGeneratedDt).alias("HdrStartDateGeneratedDt"),
                                                                     coalesce(df_InvoicedAdjustedRecord_1.PromotionUUID,df_InvoicedAdjustedRecord_2.PromotionUUID).alias("PromotionUUID"),
                                                                     df_InvoicedAdjustedRecord_1.IsSubscriptionAdjusted,
                                                                     df_InvoicedAdjustedRecord_2.IsSubscriptionAdjustedNotInvoiced
                                                                     )

    df_InvoiceAdjusted_fnl = df_InvoicedAdjustedRecord.join(df_InvoicedAdjustedRecord_3,(df_InvoicedAdjustedRecord.CoolSculptingID == df_InvoicedAdjustedRecord_3.CoolSculptingID) & (df_InvoicedAdjustedRecord.ShipToAccountId == df_InvoicedAdjustedRecord_3.ShipToAccountId) & (df_InvoicedAdjustedRecord.HdrStartDateGeneratedDt == df_InvoicedAdjustedRecord_3.HdrStartDateGeneratedDt) & (df_InvoicedAdjustedRecord.PromotionUUID == df_InvoicedAdjustedRecord_3.PromotionUUID),"FULL")\
                                                        .select(coalesce(df_InvoicedAdjustedRecord.ShipToAccountId,df_InvoicedAdjustedRecord_3.ShipToAccountId).alias("ShipToAccountId"),
                                                        coalesce(df_InvoicedAdjustedRecord.CoolSculptingID,df_InvoicedAdjustedRecord_3.CoolSculptingID).alias("CoolSculptingID"),
                                                        coalesce(df_InvoicedAdjustedRecord.HdrStartDateGeneratedDt,df_InvoicedAdjustedRecord_3.HdrStartDateGeneratedDt).alias("HdrStartDateGeneratedDt"),
                                                        coalesce(df_InvoicedAdjustedRecord.PromotionUUID,df_InvoicedAdjustedRecord_3.PromotionUUID).alias("PromotionUUID"),
                                                        df_InvoicedAdjustedRecord_3.IsSubscriptionAdjusted,
                                                        df_InvoicedAdjustedRecord_3.IsSubscriptionAdjustedNotInvoiced,
                                                        df_InvoicedAdjustedRecord.IsSubscriptionAdjustedFollowUp,
                                                        df_InvoicedAdjustedRecord.IsSubscriptionReAllignment,
                                                        df_InvoicedAdjustedRecord.IsSubscriptionAdjustedFee
                                                      ).distinct()

    return df_InvoiceAdjusted_fnl

In [0]:
def subscriptionAdjustedConsumerShipToSoldTo(InvoiceDate):
    #P3 Eligible Cycles
    df_ULLogs = validP3EligibleCycles(InvoiceDate)

    df_ULSource = df_ULLogs.filter(col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo')

    #Previous Susbcription details are retrieved
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                        .join(df_Account,["ShipToAccountUUID","PromotionUUID"], "INNER")\
                                        .filter(df_ConsumerSubscription.VersionID == 1)\
                                        .withColumn("LeadConsumerSubscriptionUUID",lead(col("ConsumerSubscriptionUUID")).over(Window.partitionBy(col("CoolSculptingID"),col("PromotionUUID")).orderBy(col("SubscriptionEndDate").asc())))\
                                        .withColumn("LeadSubscriptionStartDate",lead(col("SubscriptionStartDate")).over(Window.partitionBy(col("CoolSculptingID"),col("PromotionUUID")).orderBy(col("SubscriptionEndDate").asc())))\
                                        .select(df_Account.ShipToAccountId,
                                            df_ConsumerSubscription.SoldToAccountID,
                                            df_ConsumerSubscription.CoolSculptingID,
                                            df_ConsumerSubscription.PromotionUUID,
                                            df_ConsumerSubscription.VersionID,
                                            df_ConsumerSubscription.SubscriptionStartDate,
                                            df_ConsumerSubscription.SubscriptionEndDate,
                                            df_ConsumerSubscription.PilotSubscriptionFlag,
                                            df_ConsumerSubscription.InvoiceExceptionFlag,
                                            df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                            df_ConsumerSubscription.PreviousConsumerSubscriptionUUID,
                                            df_ConsumerSubscription.UpdatedBy,
                                            "LeadConsumerSubscriptionUUID",
                                            "LeadSubscriptionStartDate"
                                        )
    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()
    
    # Fetch Previous Version Subscription details to find the Subscription Adjustment.
    df_PreviousConsumerSubscription_2 = spark.read.table('Promotion.fact_consumersubscription')\
                                            .filter(col('VersionID') == 2)\
                                            .select(col("ConsumerSubscriptionUUID").alias("ConsumerSubscriptionUUID_2"),
                                                    col("PreviousConsumerSubscriptionUUID").alias("PreviousConsumerSubscriptionUUID_2"),
                                                    col("SubscriptionStartDate").alias("SubscriptionStartDate_2"),
                                                    col("SubscriptionEndDate").alias("SubscriptionEndDate_2"),
                                                    col("Comments").alias("Comments_2"))
    
    df_ConsumerSubscription_2 = (df_PreviousConsumerSubscription_2.alias("Subscription_2").join(df_PreviousConsumerSubscription_2.alias("PreviousSubscription_2"),(col("Subscription_2.PreviousConsumerSubscriptionUUID_2") == col("PreviousSubscription_2.ConsumerSubscriptionUUID_2")),"LEFT")
                                 .select(col("Subscription_2.ConsumerSubscriptionUUID_2"),
                                         col("Subscription_2.PreviousConsumerSubscriptionUUID_2"),
                                         col("Subscription_2.SubscriptionStartDate_2"),
                                         col("Subscription_2.SubscriptionEndDate_2"),
                                         col("Subscription_2.Comments_2"),
                                         col("PreviousSubscription_2.SubscriptionStartDate_2").alias("PreviousSubscriptionStartDate_2"),
                                         col("PreviousSubscription_2.SubscriptionEndDate_2").alias("PreviousSubscriptionEndDate_2")))
                                            
    # Join Ullogs with Consumer Subscription and join the previous version Subscription  details  
    # If the Treatment Date was lesser than the previous version subscriptionStatDate and equal to the Current SubscriptionStartDate then the record is considered as IsSubscriptionAdjustedFollowUp
    # If new Subscription Started after the Current Subscription then the record is considered as IsSubscriptionReAllignment
    # If Exception was created after TruUp Scenario, check the previous subscription of version 2 and if PreviousSubscriptionDate was greater than the Current SubscriptionStartDate then the record is considered as IsSubscriptionAdjustedFollowUp

    df_InvoicedAdjustedRecord = df_ULSource.alias("ULLogs").join(df_ConsumerSubscription_shipTo.alias("Subscription"),((col("ULLogs.ShipToAccountId") == col("Subscription.ShipToAccountId")) & (col("ULLogs.CoolSculptingID") == col("Subscription.CoolSculptingID")) & (col("ULLogs.HdrStartDateGeneratedDt").between(col("Subscription.SubscriptionStartDate"),col("Subscription.SubscriptionEndDate"))) & (col("ULLogs.PromotionUUID") == col("Subscription.PromotionUUID"))) | ((col("ULLogs.ShipToAccountId") != col("Subscription.ShipToAccountId")) & (col("ULLogs.SoldToAccountID") == col("Subscription.SoldToAccountID")) & (col("ULLogs.CoolSculptingID") == col("Subscription.CoolSculptingID")) & (col("ULLogs.HdrStartDateGeneratedDt").between(col("Subscription.SubscriptionStartDate"),col("Subscription.SubscriptionEndDate"))) & (col("ULLogs.PromotionUUID") == col("Subscription.PromotionUUID"))),"LEFT")\
                    .join(df_ConsumerSubscription_2.alias("Subscription_2"),(col("Subscription.PreviousConsumerSubscriptionUUID") == col("Subscription_2.ConsumerSubscriptionUUID_2")),"LEFT")\
                    .withColumn("IsSubscriptionAdjustedFollowUp",when(((col("ULLogs.HdrStartDateGeneratedDt") < col("Subscription_2.SubscriptionStartDate_2")) | ((col("Subscription.InvoiceExceptionFlag") == 'Y') & (col("ULLogs.HdrStartDateGeneratedDt") < col("Subscription_2.PreviousSubscriptionStartDate_2")))) & (col("ULLogs.HdrStartDateGeneratedDt") == col("Subscription.SubscriptionStartDate")),lit(1)))\
                    .withColumn("IsSubscriptionReAllignment",when((col("IsSubscriptionAdjustedFollowUp") == 1) & (col("Subscription.LeadConsumerSubscriptionUUID").isNotNull()),lit(1)))\
                    .withColumn("IsSubscriptionAdjustedFee",when((col("ULLogs.HdrStartDateGeneratedDt") == col("Subscription_2.SubscriptionStartDate_2")) | (col("ULLogs.HdrStartDateGeneratedDt") == col("Subscription_2.PreviousSubscriptionStartDate_2")),lit(1)))\
                            .select(col("ULLogs.ShipToAccountId"),
                                            col("ULLogs.SoldToAccountID"),
                                            col("ULLogs.CoolSculptingID"),
                                            col("ULLogs.PromotionUUID"),
                                            col("ULLogs.HdrStartDateGeneratedDt"),
                                            col("ULLogs.CalendarEndTimeStamp"),
                                            col("ULLogs.CalendarStartTimeStamp"),
                                            col("ULLogs.UL_CreatedDt"),
                                            col("ULLogs.TotalCycleCount"),
                                            col("Subscription.SubscriptionStartDate"),
                                            col("Subscription.SubscriptionEndDate"),
                                            col("Subscription.ConsumerSubscriptionUUID"),
                                            col("Subscription.PreviousConsumerSubscriptionUUID"),
                                            col("Subscription_2.ConsumerSubscriptionUUID_2"),
                                            col("Subscription_2.PreviousConsumerSubscriptionUUID_2"),
                                            col("Subscription_2.SubscriptionStartDate_2"),
                                            col("Subscription_2.SubscriptionEndDate_2"),
                                            col("Subscription_2.Comments_2"),
                                            "IsSubscriptionAdjustedFollowUp",
                                            "IsSubscriptionReAllignment",
                                            "IsSubscriptionAdjustedFee",
                                            col("ULLogs.PlanType")
                                            )
    
    df_InvoicedAdjustedRecordTemp = df_InvoicedAdjustedRecord.filter("IsSubscriptionAdjustedFollowUp == 1")\
                                                             .withColumnRenamed("IsSubscriptionAdjustedFollowUp","IsSubscriptionAdjustedFollowUp_1")\
                                                             .withColumnRenamed("CoolSculptingID","CoolSculptingID_1")

    #Find the already Invoiced SubscriptionAdjusted records
    df_InvoicedAdjustedRecord_1 = df_InvoicedAdjustedRecordTemp.alias("FollowUp").join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(df_InvoicedAdjustedRecordTemp.CoolSculptingID_1 == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & ((df_InvoicedAdjustedRecordTemp.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) | (df_InvoicedAdjustedRecordTemp.SoldToAccountID == df_invoiceaddendum_invoiceaddendumdetails.SoldToAccountID)) & (df_InvoicedAdjustedRecordTemp.HdrStartDateGeneratedDt < df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_InvoicedAdjustedRecordTemp.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                    .select(col("Invoiced.ShipToAccountId"),
                                            col("Invoiced.CoolSculptingID"),
                                            col("Invoiced.CycleDate").alias("HdrStartDateGeneratedDt"),
                                            col("Invoiced.PatientClassificationName"),
                                            col("Invoiced.PreviousInvoiceAddendumDetailsUUID").alias("InvoiceAddendumDetailsUUID"),
                                            col("Invoiced.IsInvoiced"),
                                            col("Invoiced.PromotionUUID"),
                                            when(col("Invoiced.PreviousInvoiceAddendumDetailsUUID").isNotNull(),lit(1)).alias("IsSubscriptionAdjusted")
                                            )\
                                    .filter(col("Invoiced.ShipToAccountId").isNotNull())

    #Find the current Invoice SubscriptionAdjusted records
    df_InvoicedAdjustedRecord_2 = df_InvoicedAdjustedRecordTemp.alias("src").join(df_InvoicedAdjustedRecord.alias("tgt"),(df_InvoicedAdjustedRecordTemp.CoolSculptingID_1 == df_InvoicedAdjustedRecord.CoolSculptingID) & (df_InvoicedAdjustedRecord.IsSubscriptionAdjustedFollowUp.isNull() & (df_InvoicedAdjustedRecordTemp.PromotionUUID == df_InvoicedAdjustedRecord.PromotionUUID)),"LEFT")\
                                                                .select(col("tgt.ShipToAccountId"),
                                                                        col("tgt.CoolSculptingID"),
                                                                        col("tgt.HdrStartDateGeneratedDt"),
                                                                        col("tgt.PromotionUUID"),
                                                                        when(col("tgt.ShipToAccountId").isNotNull(),lit(1)).alias("IsSubscriptionAdjustedNotInvoiced"))\
                                                                .filter(col("tgt.ShipToAccountId").isNotNull())
    
    #Combine Both the Invoiced/NotInvoiced SubscriptionsAdjusted records
    df_InvoicedAdjustedRecord_3 = df_InvoicedAdjustedRecord_1.join(df_InvoicedAdjustedRecord_2,(df_InvoicedAdjustedRecord_1.ShipToAccountId == df_InvoicedAdjustedRecord_2.ShipToAccountId) & (df_InvoicedAdjustedRecord_1.CoolSculptingID == df_InvoicedAdjustedRecord_2.CoolSculptingID) & (df_InvoicedAdjustedRecord_1.HdrStartDateGeneratedDt == df_InvoicedAdjustedRecord_2.HdrStartDateGeneratedDt) & (df_InvoicedAdjustedRecord_1.PromotionUUID == df_InvoicedAdjustedRecord_2.PromotionUUID),"FULL")\
                                                             .select(coalesce(df_InvoicedAdjustedRecord_1.ShipToAccountId,df_InvoicedAdjustedRecord_2.ShipToAccountId).alias("ShipToAccountId"),
                                                                     coalesce(df_InvoicedAdjustedRecord_1.CoolSculptingID,df_InvoicedAdjustedRecord_2.CoolSculptingID).alias("CoolSculptingID"),
                                                                     coalesce(df_InvoicedAdjustedRecord_1.HdrStartDateGeneratedDt,df_InvoicedAdjustedRecord_2.HdrStartDateGeneratedDt).alias("HdrStartDateGeneratedDt"),
                                                                     coalesce(df_InvoicedAdjustedRecord_1.PromotionUUID,df_InvoicedAdjustedRecord_2.PromotionUUID).alias("PromotionUUID"),
                                                                     df_InvoicedAdjustedRecord_1.IsSubscriptionAdjusted,
                                                                     df_InvoicedAdjustedRecord_2.IsSubscriptionAdjustedNotInvoiced
                                                                     )

    df_InvoiceAdjusted_fnl = df_InvoicedAdjustedRecord.join(df_InvoicedAdjustedRecord_3,(df_InvoicedAdjustedRecord.CoolSculptingID == df_InvoicedAdjustedRecord_3.CoolSculptingID) & (df_InvoicedAdjustedRecord.ShipToAccountId == df_InvoicedAdjustedRecord_3.ShipToAccountId) & (df_InvoicedAdjustedRecord.HdrStartDateGeneratedDt == df_InvoicedAdjustedRecord_3.HdrStartDateGeneratedDt) & (df_InvoicedAdjustedRecord.PromotionUUID == df_InvoicedAdjustedRecord_3.PromotionUUID),"FULL")\
                                                        .select(coalesce(df_InvoicedAdjustedRecord.ShipToAccountId,df_InvoicedAdjustedRecord_3.ShipToAccountId).alias("ShipToAccountId"),
                                                        coalesce(df_InvoicedAdjustedRecord.CoolSculptingID,df_InvoicedAdjustedRecord_3.CoolSculptingID).alias("CoolSculptingID"),
                                                        coalesce(df_InvoicedAdjustedRecord.HdrStartDateGeneratedDt,df_InvoicedAdjustedRecord_3.HdrStartDateGeneratedDt).alias("HdrStartDateGeneratedDt"),
                                                        coalesce(df_InvoicedAdjustedRecord.PromotionUUID,df_InvoicedAdjustedRecord_3.PromotionUUID).alias("PromotionUUID"),
                                                        df_InvoicedAdjustedRecord_3.IsSubscriptionAdjusted,
                                                        df_InvoicedAdjustedRecord_3.IsSubscriptionAdjustedNotInvoiced,
                                                        df_InvoicedAdjustedRecord.IsSubscriptionAdjustedFollowUp,
                                                        df_InvoicedAdjustedRecord.IsSubscriptionReAllignment,
                                                        df_InvoicedAdjustedRecord.IsSubscriptionAdjustedFee
                                                      ).distinct()

    return df_InvoiceAdjusted_fnl

In [0]:
def patientClassificationConsumerBased(df_ullogsfnl,InvoiceDate):

    df_ullogsfnl = (df_ullogsfnl.withColumn("IsPerPatientFee",row_number().over(Window.partitionBy("CoolSculptingID","PromotionUUID").orderBy(col("MinUL_CreatedDt").asc(),col("HdrStartDateGeneratedDt").asc(),col("Min_HdrStartTimeGeneratedTmstmp").asc())))
                                .filter(col("InvoiceDate")==lit(f'{InvoiceDate}')))

    #Already Invoice Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #TruUp records
    df_InvoiceAdjusted_fnl = subscriptionAdjustedConsumer(InvoiceDate)

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    "PromotionUUID",
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")

    #join ULLogs with ConsumerSubscription and fact_invoiceaddendum_invoiceaddendumdetails
    df_ULSourceSubscription = df_ullogsfnl.alias("ul").join(df_ConsumerSubscription_shipTo.alias("sub"),((col("ul.CoolSculptingID") == col("sub.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt").between(col("sub.SubscriptionStartDate"),col("sub.SubscriptionEndDate")))) & (col("ul.PromotionUUID") == col("sub.PromotionUUID")),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails.alias("inv"),(col("ul.ShipToAccountId") == col("inv.ShipToAccountId") ) & (col("ul.CoolSculptingID") == col("inv.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt") == col("inv.CycleDate")) & (col("ul.PromotionUUID") == col("inv.PromotionUUID")),"LEFT")\
                .join(df_InvoiceAdjusted_fnl.alias("adj"),(col("ul.ShipToAccountId") == col("adj.ShipToAccountId") ) & (col("ul.CoolSculptingID") == col("adj.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt") == col("adj.HdrStartDateGeneratedDt")) & (col("ul.PromotionUUID") == col("adj.PromotionUUID")),"LEFT")\
                .withColumn("IsDifferentShipTo",lit(0))\
                .withColumn("IsNewSubscription",when((col("ul.HdrStartDateGeneratedDt") == col("sub.SubscriptionStartDate")) & (col("ul.ShipToAccountId") == col("sub.ShipToAccountId")),1))\
                .select(col("ul.ShipToAccountId"),
                        col("ul.CoolSculptingID"),
                        col("ul.PromotionUUID"),
                        col("ul.HdrStartDateGeneratedDt"),
                        col("ul.MinUL_CreatedDt"),
                        col("ul.MaxUL_CreatedDt"),
                        col("ul.IsPerPatientFee"),
                        col("ul.FlagType"),
                        col("ul.CalendarStartTimeStamp"),
                        col("ul.CalendarEndTimeStamp"),
                        col("ul.PromotionRuleType"),
                        col("inv.IsInvoiced"),
                        col("sub.ConsumerSubscriptionUUID"),
                        col("sub.SubscriptionStartDate"),
                        col("sub.SubscriptionEndDate"),
                        col("adj.IsSubscriptionAdjustedFollowUp"),
                        col("adj.IsSubscriptionAdjusted"),
                        col("adj.IsSubscriptionReAllignment"),
                        col("adj.IsSubscriptionAdjustedNotInvoiced"),
                        col("ul.PlanType"),
                        "IsDifferentShipTo",
                        "IsNewSubscription",
                        col("ul.IsValidEfficacy"),
                        col("ul.IsMultipleEfficacy"))

    # If multiple subscriptions are there for a coolsculptingid, pick the subscription that was having furthest Subscription end date
    windowSpec  = Window.partitionBy("CoolSculptingID","PromotionUUID","ShipToAccountId","HdrStartDateGeneratedDt","PlanType","IsValidEfficacy").orderBy(desc("SubscriptionEndDate"))
    df_ULSourceSubscription = df_ULSourceSubscription.withColumn("IsMultiSubscripition",row_number().over(windowSpec)).filter(col("IsMultiSubscripition") == 1)

    #Verify Subscription was Invoiced
    df_source = df_ULSourceSubscription.alias("ULLogs").join(df_invoiceaddendum_invoiceaddendumdetails.alias("InvoiceAddendum"),(df_ULSourceSubscription.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_invoiceaddendum_invoiceaddendumdetails.CycleDate.between(df_ULSourceSubscription.SubscriptionStartDate,df_ULSourceSubscription.SubscriptionEndDate)) & (df_ULSourceSubscription.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
        .withColumn("IsSubscriptionInvoiced",when((col("InvoiceAddendum.IsInvoiced") == 1),1))\
        .select(df_ULSourceSubscription.ShipToAccountId,
                        df_ULSourceSubscription.CoolSculptingID,
                        df_ULSourceSubscription.HdrStartDateGeneratedDt,
                        df_ULSourceSubscription.PromotionUUID,
                        df_ULSourceSubscription.MinUL_CreatedDt,
                        df_ULSourceSubscription.MaxUL_CreatedDt,
                        df_ULSourceSubscription.IsPerPatientFee,
                        df_ULSourceSubscription.FlagType,
                        df_ULSourceSubscription.IsInvoiced,
                        df_ULSourceSubscription.ConsumerSubscriptionUUID,
                        df_ULSourceSubscription.SubscriptionStartDate,
                        df_ULSourceSubscription.SubscriptionEndDate,
                        df_ULSourceSubscription.IsDifferentShipTo,
                        df_ULSourceSubscription.IsNewSubscription,
                        df_ULSourceSubscription.CalendarStartTimeStamp,
                        df_ULSourceSubscription.CalendarEndTimeStamp,
                        df_ULSourceSubscription.IsSubscriptionAdjustedFollowUp,
                        df_ULSourceSubscription.IsSubscriptionAdjusted,
                        df_ULSourceSubscription.IsSubscriptionReAllignment,
                        df_ULSourceSubscription.IsSubscriptionAdjustedNotInvoiced,
                        df_ULSourceSubscription.PromotionRuleType,
                        df_ULSourceSubscription.PlanType,
                        df_ULSourceSubscription.IsValidEfficacy,
                        df_ULSourceSubscription.IsMultipleEfficacy,
                        "IsSubscriptionInvoiced").distinct()
        
    return df_source

In [0]:
def patientClassificationConsumerShipToSoldToBased(df_ullogsfnl,InvoiceDate):

    df_ullogsfnl = (df_ullogsfnl.withColumn("IsPerPatientFee",row_number().over(Window.partitionBy("ShipToAccountId","CoolSculptingID","PromotionUUID").orderBy(col("MinUL_CreatedDt").asc(),col("HdrStartDateGeneratedDt").asc(),col("Min_HdrStartTimeGeneratedTmstmp").asc())))
                                .filter(col("InvoiceDate")==lit(f'{InvoiceDate}')))
    
    #Already Invoice Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #TruUp records
    df_InvoiceAdjusted_fnl = subscriptionAdjustedConsumerShipToSoldTo(InvoiceDate)

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    "PromotionUUID",
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")

    #join ULLogs with ConsumerSubscription and fact_invoiceaddendum_invoiceaddendumdetails
    df_ULSourceSubscription = df_ullogsfnl.alias("ul").join(df_ConsumerSubscription_shipTo.alias("sub"),((col("ul.ShipToAccountId") == col("sub.ShipToAccountId") ) & (col("ul.CoolSculptingID") == col("sub.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt").between(col("sub.SubscriptionStartDate"),col("sub.SubscriptionEndDate")))) | ((col("ul.ShipToAccountId") != col("sub.ShipToAccountId") ) & (col("ul.SoldToAccountID")
        == col("sub.SoldToAccountID")) & (col("ul.CoolSculptingID") == col("sub.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt").between(col("sub.SubscriptionStartDate"),col("sub.SubscriptionEndDate")))) & (col("ul.PromotionUUID") == col("sub.PromotionUUID")),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails.alias("inv"),(col("ul.ShipToAccountId") == col("inv.ShipToAccountId") ) & (col("ul.CoolSculptingID") == col("inv.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt") == col("inv.CycleDate")) & (col("ul.PromotionUUID") == col("inv.PromotionUUID")),"LEFT")\
                .join(df_InvoiceAdjusted_fnl.alias("adj"),(col("ul.ShipToAccountId") == col("adj.ShipToAccountId") ) & (col("ul.CoolSculptingID") == col("adj.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt") == col("adj.HdrStartDateGeneratedDt")) & (col("ul.PromotionUUID") == col("adj.PromotionUUID")),"LEFT")\
                .withColumn("IsDifferentShipTo",when(((col("ul.ShipToAccountId") != col("sub.ShipToAccountId") ) & (col("ul.SoldToAccountID")
        == col("sub.SoldToAccountID")) & (col("ul.CoolSculptingID") == col("sub.CoolSculptingID")) & (col("ul.HdrStartDateGeneratedDt").between(col("sub.SubscriptionStartDate"),col("sub.SubscriptionEndDate")))),1))\
                .withColumn("IsNewSubscription",when((col("ul.HdrStartDateGeneratedDt") == col("sub.SubscriptionStartDate")),1))\
                .select(col("ul.ShipToAccountId"),
                        col("ul.CoolSculptingID"),
                        col("ul.PromotionUUID"),
                        col("ul.HdrStartDateGeneratedDt"),
                        col("ul.MinUL_CreatedDt"),
                        col("ul.MaxUL_CreatedDt"),
                        col("ul.IsPerPatientFee"),
                        col("ul.FlagType"),
                        col("ul.CalendarStartTimeStamp"),
                        col("ul.CalendarEndTimeStamp"),
                        col("ul.PromotionRuleType"),
                        col("inv.IsInvoiced"),
                        col("sub.ConsumerSubscriptionUUID"),
                        col("sub.SubscriptionStartDate"),
                        col("sub.SubscriptionEndDate"),
                        col("adj.IsSubscriptionAdjustedFollowUp"),
                        col("adj.IsSubscriptionAdjusted"),
                        col("adj.IsSubscriptionReAllignment"),
                        col("adj.IsSubscriptionAdjustedNotInvoiced"),
                        col("ul.PlanType"),
                        "IsDifferentShipTo",
                        "IsNewSubscription",
                        col("ul.IsValidEfficacy"),
                        col("ul.IsMultipleEfficacy"))
           

    #Verify Subscription was Invoiced
    df_source = df_ULSourceSubscription.alias("ULLogs").join(df_invoiceaddendum_invoiceaddendumdetails.alias("InvoiceAddendum"),(df_ULSourceSubscription.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_ULSourceSubscription.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_invoiceaddendum_invoiceaddendumdetails.CycleDate.between(df_ULSourceSubscription.SubscriptionStartDate,df_ULSourceSubscription.SubscriptionEndDate)) & (df_ULSourceSubscription.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
        .withColumn("IsSubscriptionInvoiced",when((col("InvoiceAddendum.IsInvoiced") == 1),1))\
        .select(df_ULSourceSubscription.ShipToAccountId,
                        df_ULSourceSubscription.CoolSculptingID,
                        df_ULSourceSubscription.HdrStartDateGeneratedDt,
                        df_ULSourceSubscription.PromotionUUID,
                        df_ULSourceSubscription.MinUL_CreatedDt,
                        df_ULSourceSubscription.MaxUL_CreatedDt,
                        df_ULSourceSubscription.IsPerPatientFee,
                        df_ULSourceSubscription.FlagType,
                        df_ULSourceSubscription.IsInvoiced,
                        df_ULSourceSubscription.ConsumerSubscriptionUUID,
                        df_ULSourceSubscription.SubscriptionStartDate,
                        df_ULSourceSubscription.SubscriptionEndDate,
                        df_ULSourceSubscription.IsDifferentShipTo,
                        df_ULSourceSubscription.IsNewSubscription,
                        df_ULSourceSubscription.CalendarStartTimeStamp,
                        df_ULSourceSubscription.CalendarEndTimeStamp,
                        df_ULSourceSubscription.IsSubscriptionAdjustedFollowUp,
                        df_ULSourceSubscription.IsSubscriptionAdjusted,
                        df_ULSourceSubscription.IsSubscriptionReAllignment,
                        df_ULSourceSubscription.IsSubscriptionAdjustedNotInvoiced,
                        df_ULSourceSubscription.PromotionRuleType,
                        df_ULSourceSubscription.PlanType,
                        df_ULSourceSubscription.IsValidEfficacy,
                        df_ULSourceSubscription.IsMultipleEfficacy,
                        "IsSubscriptionInvoiced").distinct()
        
    return df_source

In [0]:
def perPatientFeeValidationConsumerBased(df_ULSource,InvoiceDate):

    #Subscription Adjusted records
    df_InvoiceAdjusted_fnl = subscriptionAdjustedConsumer(InvoiceDate)

    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Fetching Invoice Exception Records
    df_InvoiceExceptionDetails = fetchInvoiceExceptionRecord(InvoiceDate)
    df_InvoiceExceptionDetails = df_InvoiceExceptionDetails.withColumn("IsException",lit(1))

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")
    
    #Join with Consumer Subscription to find the Per Patient Fee record  
    df_SourceClassification = df_ULSource.join(df_ConsumerSubscription_shipTo,((df_ULSource.ShipToAccountId == df_ConsumerSubscription_shipTo.ShipToAccountId ) & (df_ULSource.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate)) & (df_ULSource.PromotionUUID == df_ConsumerSubscription_shipTo.PromotionUUID)),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                .join(df_InvoiceExceptionDetails,(df_ULSource.ShipToAccountId == df_InvoiceExceptionDetails.ShipToAccountID) & (df_ULSource.CoolSculptingID == df_InvoiceExceptionDetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceExceptionDetails.CycleDate) & (df_ULSource.PromotionUUID == df_InvoiceExceptionDetails.PromotionUUID),"LEFT")\
                .withColumn("isPerPatinetFee",when((df_ULSource.HdrStartDateGeneratedDt == df_ConsumerSubscription_shipTo.SubscriptionStartDate),lit(1)))\
                .join(df_InvoiceAdjusted_fnl,(df_ULSource.ShipToAccountId == df_InvoiceAdjusted_fnl.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_InvoiceAdjusted_fnl.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceAdjusted_fnl.HdrStartDateGeneratedDt) & (df_ULSource.PromotionUUID == df_InvoiceAdjusted_fnl.PromotionUUID),"LEFT")\
                .withColumn("SubscriptionEndDateDrived",date_add(df_ULSource.HdrStartDateGeneratedDt,150))\
                .select(df_ULSource.ShipToAccountId,
                        df_ULSource.SoldToAccountID,
                        df_ULSource.CoolSculptingID,
                        df_ULSource.HdrStartDateGeneratedDt,
                        df_ULSource.UL_CreatedDt,
                        df_ULSource.TotalCycleCount,
                        df_ULSource.CalendarStartTimeStamp,
                        df_ULSource.CalendarEndTimeStamp,
                        df_ULSource.PromotionUUID,
                        df_ULSource.FlagType,
                        df_ULSource.PromotionRuleType,
                        df_ConsumerSubscription_shipTo.ConsumerSubscriptionUUID,
                        df_ConsumerSubscription_shipTo.SubscriptionStartDate,
                        df_ConsumerSubscription_shipTo.SubscriptionEndDate,
                        df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced.alias("IsInvoiced_ULLogs"),
                        df_InvoiceExceptionDetails.IsException,
                        df_InvoiceAdjusted_fnl.IsSubscriptionAdjustedFollowUp,
                        df_InvoiceAdjusted_fnl.IsSubscriptionAdjusted,
                        df_InvoiceAdjusted_fnl.IsSubscriptionReAllignment,
                        df_InvoiceAdjusted_fnl.IsSubscriptionAdjustedNotInvoiced,
                        "isPerPatinetFee",
                        "SubscriptionEndDateDrived",
                        df_ULSource.PlanType
                        )
    return df_SourceClassification


In [0]:
def perPatientFeeValidationConsumerShipToSoldToBased(df_ULSource,InvoiceDate):
    
    #Subscription Adjusted records
    df_InvoiceAdjusted_fnl = subscriptionAdjustedConsumerShipToSoldTo(InvoiceDate)

    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Fetching Invoice Exception Records
    df_InvoiceExceptionDetails = fetchInvoiceExceptionRecord(InvoiceDate)
    df_InvoiceExceptionDetails = df_InvoiceExceptionDetails.withColumn("IsException",lit(1))

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")
    
    #Join with Consumer Subscription to find the Per Patient Fee record  
    df_SourceClassification = df_ULSource.join(df_ConsumerSubscription_shipTo,((df_ULSource.ShipToAccountId == df_ConsumerSubscription_shipTo.ShipToAccountId ) & (df_ULSource.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate)) & (df_ULSource.PromotionUUID == df_ConsumerSubscription_shipTo.PromotionUUID)),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                .join(df_InvoiceExceptionDetails,(df_ULSource.ShipToAccountId == df_InvoiceExceptionDetails.ShipToAccountID) & (df_ULSource.CoolSculptingID == df_InvoiceExceptionDetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceExceptionDetails.CycleDate) & (df_ULSource.PromotionUUID == df_InvoiceExceptionDetails.PromotionUUID),"LEFT")\
                .withColumn("isPerPatinetFee",when(df_ULSource.HdrStartDateGeneratedDt == df_ConsumerSubscription_shipTo.SubscriptionStartDate,lit(1)))\
                .join(df_InvoiceAdjusted_fnl,(df_ULSource.ShipToAccountId == df_InvoiceAdjusted_fnl.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_InvoiceAdjusted_fnl.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceAdjusted_fnl.HdrStartDateGeneratedDt) & (df_ULSource.PromotionUUID == df_InvoiceAdjusted_fnl.PromotionUUID),"LEFT")\
                .withColumn("SubscriptionEndDateDrived",date_add(df_ULSource.HdrStartDateGeneratedDt,150))\
                .select(df_ULSource.ShipToAccountId,
                        df_ULSource.SoldToAccountID,
                        df_ULSource.CoolSculptingID,
                        df_ULSource.HdrStartDateGeneratedDt,
                        df_ULSource.UL_CreatedDt,
                        df_ULSource.TotalCycleCount,
                        df_ULSource.CalendarStartTimeStamp,
                        df_ULSource.CalendarEndTimeStamp,
                        df_ULSource.PromotionUUID,
                        df_ULSource.FlagType,
                        df_ULSource.PromotionRuleType,
                        df_ConsumerSubscription_shipTo.ConsumerSubscriptionUUID,
                        df_ConsumerSubscription_shipTo.SubscriptionStartDate,
                        df_ConsumerSubscription_shipTo.SubscriptionEndDate,
                        df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced.alias("IsInvoiced_ULLogs"),
                        df_InvoiceExceptionDetails.IsException,
                        df_InvoiceAdjusted_fnl.IsSubscriptionAdjustedFollowUp,
                        df_InvoiceAdjusted_fnl.IsSubscriptionAdjusted,
                        df_InvoiceAdjusted_fnl.IsSubscriptionReAllignment,
                        df_InvoiceAdjusted_fnl.IsSubscriptionAdjustedNotInvoiced,
                        "isPerPatinetFee",
                        "SubscriptionEndDateDrived",
                        df_ULSource.PlanType
                        )
    return df_SourceClassification

In [0]:
def perCycleFeeValidationConsumerBased(df_ULSource,InvoiceDate):
    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Fetching Invoice Exception Records
    df_InvoiceExceptionDetails = fetchInvoiceExceptionRecord(InvoiceDate)
    df_InvoiceExceptionDetails = df_InvoiceExceptionDetails.withColumn("IsException",lit(1))

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")
    
    #Join with Consumer Subscription to find the record was having existing subscription
    df_SourceClassification = df_ULSource.join(df_ConsumerSubscription_shipTo,((df_ULSource.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate)) & (df_ULSource.PromotionUUID == df_ConsumerSubscription_shipTo.PromotionUUID)),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                .join(df_InvoiceExceptionDetails,(df_ULSource.ShipToAccountId == df_InvoiceExceptionDetails.ShipToAccountID) & (df_ULSource.CoolSculptingID == df_InvoiceExceptionDetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceExceptionDetails.CycleDate) & (df_ULSource.PromotionUUID == df_InvoiceExceptionDetails.PromotionUUID),"LEFT")\
                .select(df_ULSource.ShipToAccountId,
                        df_ULSource.SoldToAccountID,
                        df_ULSource.CoolSculptingID,
                        df_ULSource.HdrStartDateGeneratedDt,
                        df_ULSource.UL_CreatedDt,
                        df_ULSource.TotalCycleCount,
                        df_ULSource.CalendarStartTimeStamp,
                        df_ULSource.CalendarEndTimeStamp,
                        df_ULSource.PromotionUUID,
                        df_ULSource.FlagType,
                        df_ULSource.PromotionRuleType,
                        df_ConsumerSubscription_shipTo.ConsumerSubscriptionUUID,
                        df_ConsumerSubscription_shipTo.SubscriptionStartDate,
                        df_ConsumerSubscription_shipTo.SubscriptionEndDate,
                        df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced.alias("IsInvoiced_ULLogs"),
                        df_InvoiceExceptionDetails.IsException,
                        df_ULSource.PlanType,
                        df_ULSource.IsValidEfficacy
                        )\
                .withColumn("InvoiceAmountDerived",df_ULSource.TotalCycleCount * 200)
    return df_SourceClassification

In [0]:
def perCycleFeeValidationConsumerShipToSoldToBased(df_ULSource,InvoiceDate):
    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Fetching Invoice Exception Records
    df_InvoiceExceptionDetails = fetchInvoiceExceptionRecord(InvoiceDate)
    df_InvoiceExceptionDetails = df_InvoiceExceptionDetails.withColumn("IsException",lit(1))

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")
    
    #Join with Consumer Subscription to find the record was having existing subscription 
    df_SourceClassification = df_ULSource.join(df_ConsumerSubscription_shipTo,(((df_ULSource.ShipToAccountId == df_ConsumerSubscription_shipTo.ShipToAccountId ) & (df_ULSource.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate))) | ((df_ULSource.ShipToAccountId != df_ConsumerSubscription_shipTo.ShipToAccountId) & (df_ULSource.SoldToAccountID == df_ConsumerSubscription_shipTo.SoldToAccountID) & (df_ULSource.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate)))) & (df_ULSource.PromotionUUID == df_ConsumerSubscription_shipTo.PromotionUUID),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                .join(df_InvoiceExceptionDetails,(df_ULSource.ShipToAccountId == df_InvoiceExceptionDetails.ShipToAccountID) & (df_ULSource.CoolSculptingID == df_InvoiceExceptionDetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceExceptionDetails.CycleDate) & (df_ULSource.PromotionUUID == df_InvoiceExceptionDetails.PromotionUUID),"LEFT")\
                .select(df_ULSource.ShipToAccountId,
                        df_ULSource.SoldToAccountID,
                        df_ULSource.CoolSculptingID,
                        df_ULSource.HdrStartDateGeneratedDt,
                        df_ULSource.UL_CreatedDt,
                        df_ULSource.TotalCycleCount,
                        df_ULSource.CalendarStartTimeStamp,
                        df_ULSource.CalendarEndTimeStamp,
                        df_ULSource.PromotionUUID,
                        df_ULSource.FlagType,
                        df_ULSource.PromotionRuleType,
                        df_ConsumerSubscription_shipTo.ConsumerSubscriptionUUID,
                        df_ConsumerSubscription_shipTo.SubscriptionStartDate,
                        df_ConsumerSubscription_shipTo.SubscriptionEndDate,
                        df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced.alias("IsInvoiced_ULLogs"),
                        df_InvoiceExceptionDetails.IsException,
                        df_ULSource.PlanType,
                        df_ULSource.IsValidEfficacy
                        )\
                .withColumn("InvoiceAmountDerived",df_ULSource.TotalCycleCount * 200)
    return df_SourceClassification


In [0]:
def offboardingrulesConsumerBased(df_P3ULLogs,InvoiceDate):
    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")

    #Already Invoiced Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Subscription Adjusted Record
    df_InvoiceAdjusted = subscriptionAdjustedConsumer(InvoiceDate)

    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle Date was already processed in previous billing period(Update Cycle Scenario)
    df_P3ULLogsInvoiced = (df_P3ULLogs.alias("P3Cycle").join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(col("P3Cycle.ShipToAccountId") == col("Invoiced.ShipToAccountId")) & (col("P3Cycle.CoolSculptingID") == col("Invoiced.CoolSculptingID")) & (col("P3Cycle.HdrStartDateGeneratedDt") == col("Invoiced.CycleDate")) & (col("P3Cycle.PromotionUUID") == col("Invoiced.PromotionUUID")),"LEFT")
                            .select(col("P3Cycle.ShipToAccountId"),
                                    col("P3Cycle.CoolSculptingID"),
                                    col("P3Cycle.SoldToAccountID"),
                                    col("P3Cycle.PromotionUUID"),
                                    col("P3Cycle.HdrStartDateGeneratedDt"),
                                    col("P3Cycle.TotalCycleCount"),
                                    col("P3Cycle.FlagType"),
                                    col("P3Cycle.UL_CreatedDt"),
                                    col("P3Cycle.CalendarStartTimeStamp"),
                                    col("P3Cycle.CalendarEndTimeStamp"),
                                    col("Invoiced.IsInvoiced"),
                                    col("P3Cycle.PlanType"),
                                    col("P3Cycle.IsValidEfficacy"),
                                    col("P3Cycle.PromotionRuleType")).distinct())

    df_P3CyclesTruUp = (df_P3ULLogsInvoiced.alias("P3Cycle").join(df_InvoiceAdjusted.alias("TruUp"),(col("P3Cycle.ShipToAccountId") == col("TruUp.ShipToAccountId")) & (col("P3Cycle.CoolSculptingID") == col("TruUp.CoolSculptingID")) & (col("P3Cycle.HdrStartDateGeneratedDt") == col("TruUp.HdrStartDateGeneratedDt")) & (col("P3Cycle.PromotionUUID") == col("TruUp.PromotionUUID")),"LEFT")
                    .select(col("P3Cycle.ShipToAccountId"),
                            col("P3Cycle.CoolSculptingID"),
                            col("P3Cycle.SoldToAccountID"),
                            col("P3Cycle.PromotionUUID"),
                            col("P3Cycle.HdrStartDateGeneratedDt"),
                            col("P3Cycle.TotalCycleCount"),
                            col("P3Cycle.FlagType"),
                            col("P3Cycle.UL_CreatedDt"),
                            col("P3Cycle.CalendarStartTimeStamp"),
                            col("P3Cycle.CalendarEndTimeStamp"),
                            col("P3Cycle.IsInvoiced"),
                            col("TruUp.IsSubscriptionAdjustedFollowUp"),
                            col("TruUp.IsSubscriptionAdjustedFee"),
                            col("TruUp.IsSubscriptionAdjustedNotInvoiced"),
                            col("TruUp.IsSubscriptionAdjusted"),
                            col("TruUp.IsSubscriptionReAllignment"),
                            col("P3Cycle.PlanType"),
                            col("P3Cycle.IsValidEfficacy"),
                            col("P3Cycle.PromotionRuleType")))

    df_P3Cycle = (df_P3CyclesTruUp.filter(df_P3CyclesTruUp.CoolSculptingID != 'MISSING')
                                .withColumn("PatientClassificationNameDerived",when(col("FlagType") == 'Fraud',"NonP3PatientFraudViolation")
                                                                            .when(col("FlagType") == 'TNC',"NonP3Fee")
                                                                            .when(col("FlagType") == 'OffBoarding_NonP3',"NonP3FeeOffboarding")
                                                                            .when(col("FlagType") == 'OffBoarding_NonP3_Treatment',"NonP3FeeOffboardingTreatment")
                                                                            )
                                .filter(col("PatientClassificationNameDerived").isNotNull()))
        
    #Join with Patinet Classification and derive Invoice Charge
    df_P3CyclePatient = (df_P3Cycle.alias("P3Cycle").join(df_PatientClassification.alias("PatientClassification"),(col("P3Cycle.PromotionUUID") == col("PatientClassification.PromotionUUID")) & (col("P3Cycle.PatientClassificationNameDerived") == col("PatientClassification.PatientClassificationName")) & (col("P3Cycle.HdrStartDateGeneratedDt").between(col("PatientClassification.EffectiveDate"),col("PatientClassification.TerminationDate"))) ,"LEFT")                          
                                    .select(col("P3Cycle.ShipToAccountId"),
                                            col("P3Cycle.SoldToAccountID"),
                                            col("P3Cycle.CoolSculptingID"),
                                            col("P3Cycle.PromotionUUID"),
                                            col("P3Cycle.HdrStartDateGeneratedDt"),
                                            col("P3Cycle.UL_CreatedDt"),
                                            col("P3Cycle.CalendarEndTimeStamp"),
                                            col("P3Cycle.CalendarStartTimeStamp"),
                                            col("P3Cycle.FlagType"),
                                            col("P3Cycle.TotalCycleCount"),
                                            col("P3Cycle.IsInvoiced"),
                                            col("P3Cycle.IsSubscriptionAdjustedFollowUp"),
                                            col("P3Cycle.IsSubscriptionAdjustedFee"),
                                            col("P3Cycle.IsSubscriptionAdjustedNotInvoiced"),
                                            col("P3Cycle.IsSubscriptionAdjusted"),
                                            col("P3Cycle.IsSubscriptionReAllignment"),
                                            col("PatientClassification.PatientClassificationName"),
                                            col("PatientClassification.DisplayName"),
                                            col("PatientClassification.ListPrice"),
                                            col("P3Cycle.PlanType"),
                                            col("P3Cycle.IsValidEfficacy"),
                                            col("P3Cycle.PromotionRuleType")))

    #Join with Consumer Subscription to verify it was having any active subscription
    df_P3Cycle_Subscription = (df_P3CyclePatient.alias("ViolationCycles").join(df_ConsumerSubscription_shipTo.alias("Subscription"),(col("ViolationCycles.CoolSculptingID") == col("Subscription.CoolSculptingID")) & (col("ViolationCycles.HdrStartDateGeneratedDt").between(col("Subscription.SubscriptionStartDate"),col("Subscription.SubscriptionEndDate"))),"LEFT")

                                            .withColumn("IsActiveSubscription",when((col("Subscription.ConsumerSubscriptionUUID").isNotNull()),1).otherwise(0))
                                            .withColumn("IsPerPatientFee",when((col("Subscription.ConsumerSubscriptionUUID").isNotNull()) & (col("ViolationCycles.HdrStartDateGeneratedDt") == col("Subscription.SubscriptionStartDate")) & (col("Subscription.ShipToAccountId") == col("ViolationCycles.ShipToAccountId")),1).otherwise(0))
                                            .withColumn("IsFollowUp",when((col("Subscription.ConsumerSubscriptionUUID").isNotNull()) & ((col("ViolationCycles.HdrStartDateGeneratedDt") != col("Subscription.SubscriptionStartDate")) | (col("IsPerPatientFee") == 0)),1).otherwise(0))
                                            .select(col("ViolationCycles.ShipToAccountId"),
                                                    col("ViolationCycles.SoldToAccountID"),
                                                    col("ViolationCycles.CoolSculptingID"),
                                                    col("ViolationCycles.PromotionUUID"),
                                                    col("ViolationCycles.HdrStartDateGeneratedDt"),
                                                    col("ViolationCycles.UL_CreatedDt"),
                                                    col("ViolationCycles.CalendarEndTimeStamp"),
                                                    col("ViolationCycles.CalendarStartTimeStamp"),
                                                    col("ViolationCycles.FlagType"),
                                                    col("ViolationCycles.TotalCycleCount"),
                                                    col("ViolationCycles.IsInvoiced"),
                                                    col("ViolationCycles.IsSubscriptionAdjustedFollowUp"),
                                                    col("ViolationCycles.IsSubscriptionAdjustedFee"),
                                                    col("ViolationCycles.IsSubscriptionAdjustedNotInvoiced"),
                                                    col("ViolationCycles.IsSubscriptionAdjusted"),
                                                    col("ViolationCycles.IsSubscriptionReAllignment"),
                                                    col("ViolationCycles.PatientClassificationName"),
                                                    col("ViolationCycles.DisplayName"),
                                                    col("ViolationCycles.ListPrice"),
                                                    "IsActiveSubscription", 
                                                    "IsPerPatientFee", 
                                                    "IsFollowUp",
                                                    col("ViolationCycles.PlanType"),
                                                    col("ViolationCycles.IsValidEfficacy"),
                                                    col("ViolationCycles.PromotionRuleType")))
    return df_P3Cycle_Subscription

In [0]:
def offboardingrulesConsumerShipToSoldToBased(df_P3ULLogs,InvoiceDate):
    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")

    #Already Invoiced Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Subscription Adjusted Record
    df_InvoiceAdjusted = subscriptionAdjustedConsumerShipToSoldTo(InvoiceDate)

    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle Date was already processed in previous billing period(Update Cycle Scenario)
    df_P3ULLogsInvoiced = (df_P3ULLogs.alias("P3Cycle").join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(col("P3Cycle.ShipToAccountId") == col("Invoiced.ShipToAccountId")) & (col("P3Cycle.CoolSculptingID") == col("Invoiced.CoolSculptingID")) & (col("P3Cycle.HdrStartDateGeneratedDt") == col("Invoiced.CycleDate")) & (col("P3Cycle.PromotionUUID") == col("Invoiced.PromotionUUID")),"LEFT")
                            .select(col("P3Cycle.ShipToAccountId"),
                                    col("P3Cycle.CoolSculptingID"),
                                    col("P3Cycle.SoldToAccountID"),
                                    col("P3Cycle.PromotionUUID"),
                                    col("P3Cycle.HdrStartDateGeneratedDt"),
                                    col("P3Cycle.TotalCycleCount"),
                                    col("P3Cycle.FlagType"),
                                    col("P3Cycle.UL_CreatedDt"),
                                    col("P3Cycle.CalendarStartTimeStamp"),
                                    col("P3Cycle.CalendarEndTimeStamp"),
                                    col("Invoiced.IsInvoiced"),
                                    col("P3Cycle.PlanType"),
                                    col("P3Cycle.IsValidEfficacy"),
                                    col("P3Cycle.PromotionRuleType")).distinct())

    df_P3CyclesTruUp = (df_P3ULLogsInvoiced.alias("P3Cycle").join(df_InvoiceAdjusted.alias("TruUp"),(col("P3Cycle.ShipToAccountId") == col("TruUp.ShipToAccountId")) & (col("P3Cycle.CoolSculptingID") == col("TruUp.CoolSculptingID")) & (col("P3Cycle.HdrStartDateGeneratedDt") == col("TruUp.HdrStartDateGeneratedDt")) & (col("P3Cycle.PromotionUUID") == col("TruUp.PromotionUUID")),"LEFT")
                    .select(col("P3Cycle.ShipToAccountId"),
                            col("P3Cycle.CoolSculptingID"),
                            col("P3Cycle.SoldToAccountID"),
                            col("P3Cycle.PromotionUUID"),
                            col("P3Cycle.HdrStartDateGeneratedDt"),
                            col("P3Cycle.TotalCycleCount"),
                            col("P3Cycle.FlagType"),
                            col("P3Cycle.UL_CreatedDt"),
                            col("P3Cycle.CalendarStartTimeStamp"),
                            col("P3Cycle.CalendarEndTimeStamp"),
                            col("P3Cycle.IsInvoiced"),
                            col("TruUp.IsSubscriptionAdjustedFollowUp"),
                            col("TruUp.IsSubscriptionAdjustedFee"),
                            col("TruUp.IsSubscriptionAdjustedNotInvoiced"),
                            col("TruUp.IsSubscriptionAdjusted"),
                            col("TruUp.IsSubscriptionReAllignment"),
                            col("P3Cycle.PlanType"),
                            col("P3Cycle.IsValidEfficacy"),
                            col("P3Cycle.PromotionRuleType")))

    df_P3Cycle = (df_P3CyclesTruUp.filter(df_P3CyclesTruUp.CoolSculptingID != 'MISSING')
                                .withColumn("PatientClassificationNameDerived",when(col("FlagType") == 'Fraud',"NonP3PatientFraudViolation")
                                                                            .when(col("FlagType") == 'TNC',"NonP3Fee")
                                                                            .when(col("FlagType") == 'OffBoarding_NonP3',"NonP3FeeOffboarding")
                                                                            .when(col("FlagType") == 'OffBoarding_NonP3_Treatment',"NonP3FeeOffboardingTreatment")
                                                                            )
                                .filter(col("PatientClassificationNameDerived").isNotNull()))
        
    #Join with Patinet Classification and derive Invoice Charge
    df_P3CyclePatient = (df_P3Cycle.alias("P3Cycle").join(df_PatientClassification.alias("PatientClassification"),(col("P3Cycle.PromotionUUID") == col("PatientClassification.PromotionUUID")) & (col("P3Cycle.PatientClassificationNameDerived") == col("PatientClassification.PatientClassificationName")) & (col("P3Cycle.HdrStartDateGeneratedDt").between(col("PatientClassification.EffectiveDate"),col("PatientClassification.TerminationDate"))) ,"LEFT")                          
                                    .select(col("P3Cycle.ShipToAccountId"),
                                            col("P3Cycle.SoldToAccountID"),
                                            col("P3Cycle.CoolSculptingID"),
                                            col("P3Cycle.PromotionUUID"),
                                            col("P3Cycle.HdrStartDateGeneratedDt"),
                                            col("P3Cycle.UL_CreatedDt"),
                                            col("P3Cycle.CalendarEndTimeStamp"),
                                            col("P3Cycle.CalendarStartTimeStamp"),
                                            col("P3Cycle.FlagType"),
                                            col("P3Cycle.TotalCycleCount"),
                                            col("P3Cycle.IsInvoiced"),
                                            col("P3Cycle.IsSubscriptionAdjustedFollowUp"),
                                            col("P3Cycle.IsSubscriptionAdjustedFee"),
                                            col("P3Cycle.IsSubscriptionAdjustedNotInvoiced"),
                                            col("P3Cycle.IsSubscriptionAdjusted"),
                                            col("P3Cycle.IsSubscriptionReAllignment"),
                                            col("PatientClassification.PatientClassificationName"),
                                            col("PatientClassification.DisplayName"),
                                            col("PatientClassification.ListPrice"),
                                            col("P3Cycle.PlanType"),
                                            col("P3Cycle.IsValidEfficacy"),
                                            col("P3Cycle.PromotionRuleType")))

    #Join with Consumer Subscription to verify it was having any active subscription
    df_P3Cycle_Subscription = (df_P3CyclePatient.alias("ViolationCycles").join(df_ConsumerSubscription_shipTo.alias("Subscription"),((col("ViolationCycles.ShipToAccountId") == col("Subscription.ShipToAccountId")) | ((col("ViolationCycles.SoldToAccountID") == col("Subscription.SoldToAccountID")) & (col("ViolationCycles.ShipToAccountId") != col("Subscription.ShipToAccountId")))) & (col("ViolationCycles.CoolSculptingID") == col("Subscription.CoolSculptingID")) & (col("ViolationCycles.HdrStartDateGeneratedDt").between(col("Subscription.SubscriptionStartDate"),col("Subscription.SubscriptionEndDate"))),"LEFT")

                                            .withColumn("IsActiveSubscription",when((col("Subscription.ConsumerSubscriptionUUID").isNotNull()),1).otherwise(0))
                                            .withColumn("IsFollowUp",when((col("Subscription.ConsumerSubscriptionUUID").isNotNull()) & (col("ViolationCycles.HdrStartDateGeneratedDt") != col("Subscription.SubscriptionStartDate")),1).otherwise(0))
                                            .withColumn("IsPerPatientFee",when((col("Subscription.ConsumerSubscriptionUUID").isNotNull()) & (col("ViolationCycles.HdrStartDateGeneratedDt") == col("Subscription.SubscriptionStartDate")),1).otherwise(0))
                                            .select(col("ViolationCycles.ShipToAccountId"),
                                                    col("ViolationCycles.SoldToAccountID"),
                                                    col("ViolationCycles.CoolSculptingID"),
                                                    col("ViolationCycles.PromotionUUID"),
                                                    col("ViolationCycles.HdrStartDateGeneratedDt"),
                                                    col("ViolationCycles.UL_CreatedDt"),
                                                    col("ViolationCycles.CalendarEndTimeStamp"),
                                                    col("ViolationCycles.CalendarStartTimeStamp"),
                                                    col("ViolationCycles.FlagType"),
                                                    col("ViolationCycles.TotalCycleCount"),
                                                    col("ViolationCycles.IsInvoiced"),
                                                    col("ViolationCycles.IsSubscriptionAdjustedFollowUp"),
                                                    col("ViolationCycles.IsSubscriptionAdjustedFee"),
                                                    col("ViolationCycles.IsSubscriptionAdjustedNotInvoiced"),
                                                    col("ViolationCycles.IsSubscriptionAdjusted"),
                                                    col("ViolationCycles.IsSubscriptionReAllignment"),
                                                    col("ViolationCycles.PatientClassificationName"),
                                                    col("ViolationCycles.DisplayName"),
                                                    col("ViolationCycles.ListPrice"),
                                                    "IsActiveSubscription", 
                                                    "IsPerPatientFee", 
                                                    "IsFollowUp",
                                                    col("ViolationCycles.PlanType"),
                                                    col("ViolationCycles.IsValidEfficacy"),
                                                    col("ViolationCycles.PromotionRuleType")))
    return df_P3Cycle_Subscription

In [0]:
def perCycleEfficacyFailValidationConsumerBased(df_ULSource,InvoiceDate):
    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Fetching Invoice Exception Records
    df_InvoiceExceptionDetails = fetchInvoiceExceptionRecord(InvoiceDate)
    df_InvoiceExceptionDetails = df_InvoiceExceptionDetails.withColumn("IsException",lit(1))

    #ConsumerSubscription ShipTo
    df_ConsumerSubscription_shipTo = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                            .filter(col("VersionId") == 1)\
                                                            .select("ConsumerSubscriptionUUID",
                                                                    "SoldToAccountID",
                                                                    "ShipToAccountId",
                                                                    df_ConsumerSubscription.PromotionUUID,
                                                                    "CoolSculptingID",
                                                                    "SubscriptionStartDate",
                                                                    "SubscriptionEndDate")
    
    #Join with Consumer Subscription to find the record was having existing subscription
    df_SourceClassification = df_ULSource.join(df_ConsumerSubscription_shipTo,((df_ULSource.CoolSculptingID == df_ConsumerSubscription_shipTo.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt.between(df_ConsumerSubscription_shipTo.SubscriptionStartDate,df_ConsumerSubscription_shipTo.SubscriptionEndDate)) & (df_ULSource.PromotionUUID == df_ConsumerSubscription_shipTo.PromotionUUID)),"LEFT")\
                .join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                .join(df_InvoiceExceptionDetails,(df_ULSource.ShipToAccountId == df_InvoiceExceptionDetails.ShipToAccountID) & (df_ULSource.CoolSculptingID == df_InvoiceExceptionDetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_InvoiceExceptionDetails.CycleDate) & (df_ULSource.PromotionUUID == df_InvoiceExceptionDetails.PromotionUUID),"LEFT")\
                .select(df_ULSource.ShipToAccountId,
                        df_ULSource.SoldToAccountID,
                        df_ULSource.CoolSculptingID,
                        df_ULSource.HdrStartDateGeneratedDt,
                        df_ULSource.UL_CreatedDt,
                        df_ULSource.TotalCycleCount,
                        df_ULSource.CalendarStartTimeStamp,
                        df_ULSource.CalendarEndTimeStamp,
                        df_ULSource.PromotionUUID,
                        df_ULSource.FlagType,
                        df_ULSource.PromotionRuleType,
                        df_ConsumerSubscription_shipTo.ConsumerSubscriptionUUID,
                        df_ConsumerSubscription_shipTo.SubscriptionStartDate,
                        df_ConsumerSubscription_shipTo.SubscriptionEndDate,
                        df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced.alias("IsInvoiced_ULLogs"),
                        df_InvoiceExceptionDetails.IsException,
                        df_ULSource.PlanType,
                        df_ULSource.IsValidEfficacy
                        )
    return df_SourceClassification

## Pre Validation Test Cases

In [0]:
#Validate Cycle Counts between ullogs and InvoiceAddendumDetails as well as the Cycles are falls within the Subscription period
def validateCycleCount(InvoiceDate,TestCaseName):
    
    #Eligible P3 Cycles
    df_ULSource = validP3EligibleCycles(InvoiceDate,'CycleCount')

    #Already Invoiced Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()
    
    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle Date was already processed in previous billing period(Update Cycle Scenario)
    df_source = df_ULSource.join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                            .select(df_ULSource.ShipToAccountId,
                                    df_ULSource.CoolSculptingID,
                                    df_ULSource.PromotionUUID,
                                    df_ULSource.HdrStartDateGeneratedDt,
                                    df_ULSource.TotalCycleCount,
                                    df_ULSource.UL_CreatedDt,
                                    df_ULSource.CalendarStartTimeStamp,
                                    df_ULSource.CalendarEndTimeStamp,
                                    df_ULSource.PromotionRuleType,
                                    df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced)
                            
    #Current Month record from FactInvoiceAddendumDetails
    df_Invoicetarget = currentInvoiceRecord(InvoiceDate,'CycleCount')
                                    
    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle information was already processed(Invoice Details could be processed in previous billing period and later exception was created)
    df_target = df_Invoicetarget.join(df_invoiceaddendum_invoiceaddendumdetails,(df_Invoicetarget.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_Invoicetarget.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_Invoicetarget.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_Invoicetarget.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
            .select(df_Invoicetarget.ShipToAccountId,
                    df_Invoicetarget.CoolSculptingID,
                    df_Invoicetarget.PromotionUUID,
                    df_Invoicetarget.CycleDate,
                    df_Invoicetarget.CycleCount,
                    df_Invoicetarget.CalendarStartTimeStamp,
                    df_Invoicetarget.CalendarEndTimeStamp,
                    df_Invoicetarget.CreatedBy.alias("Details_CreatedBy"),
                    df_Invoicetarget.Details_UpdatedDt,
                    df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced,
                    "Details_CreatedDt",
                    "PilotSubscriptionFlag")

    #Add prefix in both ullogs and invoiceaddendumdetails dataframes column names
    df_source = add_Prefix_ColumnName(df_source,"ULLogs_")
    df_target = add_Prefix_ColumnName(df_target,"InvoiceAddendumDetails_")

    #Comparing the ULLogs and FactInvoiceAddendumDetails
    df_result = df_source.alias("Src").join(df_target.alias("Tgt"),(df_source.ULLogs_ShipToAccountId == df_target.InvoiceAddendumDetails_ShipToAccountId) & (df_source.ULLogs_CoolSculptingID == df_target.InvoiceAddendumDetails_CoolSculptingID)  & (df_source.ULLogs_HdrStartDateGeneratedDt == df_target.InvoiceAddendumDetails_CycleDate) & (df_source.ULLogs_PromotionUUID == df_target.InvoiceAddendumDetails_PromotionUUID),"FULL")\
                .withColumn("Comments",when(col("Src.ULLogs_TotalCycleCount") == col("Tgt.InvoiceAddendumDetails_CycleCount"), "Matched")
                                        .when((col("Src.ULLogs_IsInvoiced").isNotNull()) | (col("Tgt.InvoiceAddendumDetails_IsInvoiced").isNotNull()), "Already Processed into Previous Billing Period")
                                        .when((col("Src.ULLogs_UL_CreatedDt").between(col("Src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("Src.ULLogs_CalendarEndTimeStamp"))),"Cycle Received within 20 minutes of Billing Period End Date")\
                                        .when((col("Tgt.InvoiceAddendumDetails_Details_CreatedDt").between(col("Tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("Tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))),"Cycle Received within 20 minutes of Billing Period Start Date")\
                                        .when((col("Tgt.InvoiceAddendumDetails_PilotSubscriptionFlag") == 'Y') & (col("Tgt.InvoiceAddendumDetails_CycleDate") <= '2024-03-28'),"Pilot Subscription")\
                                        .when((col("Src.ULLogs_UL_CreatedDt").isNull()) & (col("Tgt.InvoiceAddendumDetails_Details_CreatedBy").like('AAIOT%')),"Record was added as part of Data Patch")
                                        .when((col("Tgt.InvoiceAddendumDetails_Details_UpdatedDt").between(col("Tgt.InvoiceAddendumDetails_CalendarEndTimeStamp"),col("Tgt.InvoiceAddendumDetails_CalendarEndTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))) & (col("Src.ULLogs_TotalCycleCount") < col("Tgt.InvoiceAddendumDetails_CycleCount")),"Additional Cycles are received after the billing period end date")
                                        .otherwise("Unknown"))

    ExceptionCount = df_result.filter("Comments == 'Unknown'").count()  
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comparison between ULLogs and FactInvoiceAddendumDetails")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
def validateTreatmentDates(InvoiceDate,TestCaseName):

    #Eligible P3 Cycles
    df_ULSource = validP3EligibleCycles(InvoiceDate,'CycleCount')

    #Already Invoiced Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()
       
    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle Date was already processed in previous billing period(Delayed Cycles)
    df_source = df_ULSource.join(df_invoiceaddendum_invoiceaddendumdetails,(df_ULSource.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_ULSource.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_ULSource.HdrStartDateGeneratedDt == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_ULSource.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                        .select(df_ULSource.ShipToAccountId,
                                df_ULSource.CoolSculptingID,
                                df_ULSource.PromotionUUID,
                                df_ULSource.HdrStartDateGeneratedDt,
                                df_ULSource.TotalCycleCount,
                                df_ULSource.UL_CreatedDt,
                                df_ULSource.CalendarStartTimeStamp,
                                df_ULSource.CalendarEndTimeStamp,
                                df_ULSource.PromotionRuleType,
                                "PreviousInvoiceAddendumDetailsUUID")

    #Current Month record from FactInvoiceAddendumDetails
    df_Invoicetarget = currentInvoiceRecord(InvoiceDate,'CycleCount')
                                
    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle information was already processed(Invoice Details could be processed in previous billing period and later exception was created)
    df_target = df_Invoicetarget.join(df_invoiceaddendum_invoiceaddendumdetails,(df_Invoicetarget.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_Invoicetarget.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_Invoicetarget.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate)  & (df_Invoicetarget.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
        .select(df_Invoicetarget.ShipToAccountId,
                df_Invoicetarget.CoolSculptingID,
                df_Invoicetarget.PromotionUUID,
                df_Invoicetarget.CycleDate.alias("TargetCycleDate"),
                df_Invoicetarget.CycleCount,
                df_Invoicetarget.CalendarStartTimeStamp,
                df_Invoicetarget.CalendarEndTimeStamp,
                df_Invoicetarget.CreatedBy.alias("Details_CreatedBy"),
                "PreviousInvoiceAddendumDetailsUUID",
                "Details_CreatedDt",
                "PilotSubscriptionFlag")

    #Add prefix in both ullogs and invoiceaddendumdetails dataframes column names
    df_source = add_Prefix_ColumnName(df_source,"ULLogs_")
    df_target = add_Prefix_ColumnName(df_target,"InvoiceAddendumDetails_")

    #Comparing the ULLogs and FactInvoiceAddendumDetails
    df_result = df_source.alias("Src").join(df_target.alias("Tgt"),(df_source.ULLogs_ShipToAccountId == df_target.InvoiceAddendumDetails_ShipToAccountId) & (df_source.ULLogs_CoolSculptingID == df_target.InvoiceAddendumDetails_CoolSculptingID)  & (df_source.ULLogs_HdrStartDateGeneratedDt == df_target.InvoiceAddendumDetails_TargetCycleDate) & (df_source.ULLogs_PromotionUUID == df_target.InvoiceAddendumDetails_PromotionUUID),"FULL")\
                .withColumn("Comments",when(df_source.ULLogs_HdrStartDateGeneratedDt == df_target.InvoiceAddendumDetails_TargetCycleDate, "Matched")
                                        .when((col("Src.ULLogs_PreviousInvoiceAddendumDetailsUUID").isNotNull()) | (col("Tgt.InvoiceAddendumDetails_PreviousInvoiceAddendumDetailsUUID").isNotNull()), "Already Processed into Previous Billing Period")\
                                        .when((col("Src.ULLogs_UL_CreatedDt").between(col("Src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("Src.ULLogs_CalendarEndTimeStamp"))),"Cycle Received within 20 minutes of Billing Period End Date")\
                                        .when((col("Tgt.InvoiceAddendumDetails_Details_CreatedDt").between(col("Tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("Tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))),"Cycle Received within 20 minutes of Billing Period Start Date")\
                                        .when((col("Tgt.InvoiceAddendumDetails_TargetCycleDate") <= '2024-03-28') & (col("Tgt.InvoiceAddendumDetails_PilotSubscriptionFlag") == 'Y'),"Cycle Received before March 2024")\
                                        .when((col("Src.ULLogs_UL_CreatedDt").isNull()) & (col("Tgt.InvoiceAddendumDetails_Details_CreatedBy").like('AAIOT%')),"Record was added as part of Data Patch")
                                        .otherwise("Unknown"))                       
    ExceptionCount = df_result.filter("Comments == 'Unknown'").count()  
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comparison between ULLogs and FactInvoiceAddendumDetails")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
#Validate exceptions received from Moxie are processed into invoice addendum details table
def validateExceptionProcessed(InvoiceDate,TestCaseName):

    df_InvoiceException = (spark.read.table("Promotion.fact_invoiceexception")
        .withColumnRenamed("PhoneNumber", "CoolSculptingID")
        .filter(((col("IsProcessedFlag") == True) | (col("IsProcessedFlag").isNull())) & (col("ConsumerSubscriptionEndDate").isNull())))

    df_InvoiceAddendumDetails_Patient = (df_InvoiceAddendumDetails.join(df_PatientClassification,df_InvoiceAddendumDetails.PatientClassificationUUID
    == df_PatientClassification.PatientClassificationUUID,"INNER")
                                        .filter(df_PatientClassification.PatientClassificationName.isin('NoPatientAssociationFeeExceptionPostInvoice','NoPatientAssociationFeeExceptionPreInvoice','NoPatientAssociationFeePerCycleException','PerPatientFeeExceptionPostInvoice','PerPatientFeeExceptionPreInvoice','PerCycleFeeExceptionPostInvoice','PerCycleFeeExceptionPreInvoice','PerPatientFeeException','Exception'))
                                        .select(df_InvoiceAddendumDetails.PromotionUUID,
                                                df_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                df_InvoiceAddendumDetails.ExceptionMoxieCaseNumber)
                                        )
    df_result = (df_InvoiceException.join(df_InvoiceCycleMonth,(df_InvoiceException.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f"{InvoiceDate}")),"INNER")
                                    .join(df_ConsumerSubscription_ShipTo,(df_ConsumerSubscription_ShipTo.ShipToAccountId == df_InvoiceException.ShipToAccountID)& (df_InvoiceException.CoolSculptingID == df_ConsumerSubscription_ShipTo.CoolSculptingID) & (df_InvoiceException.CycleDate.between(df_ConsumerSubscription_ShipTo.SubscriptionStartDate,df_ConsumerSubscription_ShipTo.SubscriptionEndDate)) & (df_ConsumerSubscription_ShipTo.VersionID == 1) & (df_ConsumerSubscription_ShipTo.PromotionUUID == df_InvoiceException.PromotionUUID),"LEFT")
                                    .join(df_InvoiceAddendumDetails_Patient,(df_InvoiceAddendumDetails_Patient.ExceptionMoxieCaseNumber == df_InvoiceException.MoxieCaseNumber) & (df_InvoiceAddendumDetails_Patient.PromotionUUID == df_InvoiceException.PromotionUUID),"LEFT")
                                    .select(df_InvoiceException.ShipToAccountID,
                                            df_InvoiceException.CoolSculptingID.alias("PhoneNumber"),
                                            df_InvoiceException.PromotionUUID,
                                            df_InvoiceException.CycleDate,
                                            df_InvoiceException.CycleCount,
                                            df_InvoiceException.MoxieCaseNumber,
                                            df_InvoiceException.IncrementalInvoiceCharge,
                                            df_InvoiceException.Comments.alias("ExceptionComments"),
                                            df_InvoiceException.IsProcessedFlag,
                                            df_InvoiceException.CreatedDate.alias("ExceptionCreatedDate"),
                                            df_InvoiceException.UpdatedDate.alias("ExceptionUpdatedDate"),
                                            df_ConsumerSubscription_ShipTo.SubscriptionStartDate,
                                            df_ConsumerSubscription_ShipTo.SubscriptionEndDate,
                                            df_ConsumerSubscription_ShipTo.ConsumerSubscriptionUUID,
                                            df_ConsumerSubscription_ShipTo.UpdatedBy,
                                            df_ConsumerSubscription_ShipTo.PilotSubscriptionFlag,
                                            df_InvoiceAddendumDetails_Patient.InvoiceAddendumDetailsUUID,
                                            df_InvoiceAddendumDetails_Patient.ExceptionMoxieCaseNumber,
                                            df_InvoiceAddendumDetails_Patient.PromotionUUID,
                                            df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                            df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                            df_InvoiceCycleMonth.InvoiceDate
                                            )
                                            .withColumn("SubscriptionEndDate_Derived",when(col("SubscriptionEndDate") > col("ExceptionCreatedDate"),col("ExceptionUpdatedDate")).otherwise(col("SubscriptionEndDate")))
                                            .withColumn("Comments",when(col("MoxieCaseNumber") == col("ExceptionMoxieCaseNumber"),"Exception Processed in Rules Engine")
                                                                .when(((col("ShipToAccountID") == "null")), "Invalid ShipToAccountId")
                                                                .when((col("IncrementalInvoiceCharge") == "950")& (~col("CycleDate").between(col("SubscriptionStartDate"), col("SubscriptionEndDate"))),"Cycle Date Not falls within the Subscription period")
                                                                .when(df_ConsumerSubscription_ShipTo.UpdatedBy == "Replication_Subscription","Subscription was replicated from One ShipToAccountId to Another ShipToAccountId, It's a valid Subscription")
                                                                .when((col("ExceptionUpdatedDate")>= col("CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"))& ((col("IsProcessedFlag") == True) | (col("IsProcessedFlag").isNull())),"Exception was received Recently")
                                                                .when((df_ConsumerSubscription_ShipTo.PilotSubscriptionFlag == "Y" ),"Subscription is either a Pilot or Replication Subscription")
                                                                .otherwise("Unknown"))
    )
                                    
    ExceptionCount = df_result.filter("Comments == 'Unknown'").count() 
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Exception Received for this Invoice Month")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
#For per patient exceptions validate if the subscription has ended
def validatePerPatientFeeException(InvoiceDate,TestCaseName):

    df_InvoiceException = (spark.read.table("Promotion.fact_invoiceexception")
        .withColumnRenamed("PhoneNumber", "CoolSculptingID")
        .filter(((col("IsProcessedFlag") == True) | (col("IsProcessedFlag").isNull())) & (col("IncrementalInvoiceCharge") == "950")))
    
    df_ConsumerSubscription_previous = df_ConsumerSubscription.join(df_DIMPromotion,["PromotionUUID"], "INNER")\
                                .join(df_Account, ["ShipToAccountUUID","PromotionUUID"], "INNER")\
                                .select(df_Account.ShipToAccountId,
                                        df_ConsumerSubscription.CoolSculptingID,
                                        df_ConsumerSubscription.SoldToAccountID,
                                        df_ConsumerSubscription.VersionID,
                                        df_ConsumerSubscription.SubscriptionStartDate,
                                        df_ConsumerSubscription.SubscriptionEndDate,
                                        df_ConsumerSubscription.PilotSubscriptionFlag,
                                        df_ConsumerSubscription.ConsumerSubscriptionUUID,
                                        df_ConsumerSubscription.UpdatedBy,
                                        df_ConsumerSubscription.PromotionUUID,
                                        df_ConsumerSubscription.CreatedDate
                                    )\
                                .withColumn("PreviousSubscriptionEndDate",lag("SubscriptionEndDate").over(Window.partitionBy("CoolSculptingID","ShipToAccountId").orderBy(asc("CreatedDate"))))\
                                .filter("VersionID ==1")

    df_result = df_InvoiceException.join(df_InvoiceCycleMonth,(df_InvoiceException.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f"{InvoiceDate}")),"INNER")\
                                                .join(df_ConsumerSubscription_previous,(df_ConsumerSubscription_previous.ShipToAccountId == df_InvoiceException.ShipToAccountID) & (df_InvoiceException.CoolSculptingID == df_ConsumerSubscription_previous.CoolSculptingID) & (df_InvoiceException.CycleDate.between(df_ConsumerSubscription_previous.SubscriptionStartDate,df_ConsumerSubscription_previous.SubscriptionEndDate)) & (df_ConsumerSubscription_previous.PromotionUUID == df_InvoiceException.PromotionUUID),"LEFT")\
                                                .filter(df_ConsumerSubscription_previous.VersionID == 1)\
                                                .select(df_InvoiceException.ShipToAccountID,
                                                        df_InvoiceException.CoolSculptingID.alias("PhoneNumber"),
                                                        df_InvoiceException.PromotionUUID,
                                                        df_InvoiceException.CycleDate,
                                                        df_InvoiceException.CycleCount,
                                                        df_InvoiceException.MoxieCaseNumber,
                                                        df_InvoiceException.IncrementalInvoiceCharge,
                                                        df_InvoiceException.IsProcessedFlag,
                                                        df_InvoiceException.ConsumerSubscriptionEndDate,
                                                        df_InvoiceException.Comments.alias("ExceptionComments"),
                                                        df_InvoiceException.CreatedDate.alias("ExceptionCreatedDate"),
                                                        df_InvoiceException.UpdatedDate.alias("ExceptionUpdatedDate"),
                                                        df_ConsumerSubscription_previous.SubscriptionStartDate,
                                                        df_ConsumerSubscription_previous.SubscriptionEndDate,
                                                        df_ConsumerSubscription_previous.ConsumerSubscriptionUUID,
                                                        df_ConsumerSubscription_previous.PromotionUUID,
                                                        df_ConsumerSubscription_previous.UpdatedBy,
                                                        df_ConsumerSubscription_previous.PilotSubscriptionFlag,
                                                        df_ConsumerSubscription_previous.PreviousSubscriptionEndDate,
                                                        df_ConsumerSubscription_previous.VersionID,
                                                        df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                                        df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                                        df_InvoiceCycleMonth.InvoiceDate)\
                                                .withColumn("SubscriptionEndDate_Derived",when(col("SubscriptionEndDate") > col("ExceptionCreatedDate"), to_date(col("ExceptionUpdatedDate")))
                                                        .otherwise(to_date(col("SubscriptionEndDate"))))\
                                                .withColumn("Comments",when(col("SubscriptionEndDate") == col("SubscriptionEndDate_Derived"), "Subscription Ended Correctly")\
                                                                .when(~col("CycleDate").between(col("SubscriptionStartDate"), col("SubscriptionEndDate")), "Cycle Date Not falls within the Subscription period")\
                                                                .when(df_ConsumerSubscription_previous.UpdatedBy == 'Replication_Subscription', "Subscription was replicated from One ShipToAccountId to Another ShipToAccountId, It's a valid Subscription")\
                                                                .when((col("ExceptionUpdatedDate")>= col("CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"))& ((col("IsProcessedFlag") == True) | (col("IsProcessedFlag").isNull())),"Exception was received Recently")\
                                                                .when((df_ConsumerSubscription_previous.PilotSubscriptionFlag == "Y" ),"Subscription is either a Pilot or Replication Subscription")\
                                                                .when((df_ConsumerSubscription_previous.SubscriptionEndDate == df_InvoiceException.ConsumerSubscriptionEndDate) | (df_ConsumerSubscription_previous.PreviousSubscriptionEndDate == df_InvoiceException.ConsumerSubscriptionEndDate),"Consumer Subscription End Date Changed Correctly") 
                                                                .when(col("SubscriptionEndDate") != col("SubscriptionEndDate_Derived"), "Unknown")\
                                                        .otherwise("Unknown"))
                                            
    ExceptionCount = df_result.filter(col("Comments")== "Unknown").count()
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"No Of Per Patinet Fee Exception Subscription Received for this Invoice Month:")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
#Verify when a UL cycle is received there is a check to see if the consumer has a subscription. Is yes, then a Followup fee is applied else a Per Patient Fee is applied
def validateByPatientClassification(InvoiceDate,TestCaseName):
    
    df_ULLogs = spark.read.table(f"{ULLogsTable}")\
                    .withColumn("CoolSculptingID", when(coalesce(col("CoolSculptingID").cast("bigint"), lit(0)) == 0, "MISSING").otherwise(col("CoolSculptingID")))\
                     .withColumn("TreatmentTime",when(col("HdrStartTimeGeneratedTmstmp")>col("EventEndTmstmp"),expr("timestampdiff(MINUTE,EventEndTmstmp, HdrStartTimeGeneratedTmstmp)"))
                                .otherwise(expr("timestampdiff(MINUTE, HdrStartTimeGeneratedTmstmp, EventEndTmstmp)")))\
                    .withColumn("PlanType",when((col("PlanType").isin("ComprehensiveTreatmentPlan")) | (col("PlanType").isNull() | (trim(col("PlanType")) == "")),"ComprehensiveTreatmentPlan").otherwise(col("PlanType")))\
                    .filter(col("CoolSculptingID") != 'MISSING')

    df_ULSource = (df_ULLogs.join(broadcast(df_DIMPromotion),(df_ULLogs.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd),"INNER")
                        .join(df_AccountRank,(df_AccountRank.ShipToAccountId == df_ULLogs.ShipToAccountID) & (df_AccountRank.PromotionUUID == df_DIMPromotion.PromotionUUID),"INNER")
                        .join(df_SmartCard,(df_ULLogs.CardSPSerialNbr == df_SmartCard.SmartCardSerialNumber) & (df_DIMPromotion.PromotionUUID == df_SmartCard.PromotionUUID),"INNER")
                        .join(df_InvoiceCycleMonth,(df_ULLogs.CreatedDt.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"LEFT")
                        .join(df_applicator,(substring(df_ULLogs.ApplicatorConfigurantionNbr,1,7) == df_applicator.SSAPPIdentID),"left")
                        .filter((df_ULLogs.HdrStartDateGeneratedDt.between(df_AccountRank.EffectiveDate, df_AccountRank.TerminationDate)) & (df_ULLogs.HdrStartDateGeneratedDt.between(df_SmartCard.EffectiveDate, df_SmartCard.TerminationDate)) & (df_ULLogs.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate)) & (df_ULLogs.PlanType.isin('SmallTreatmentPlan','ComprehensiveTreatmentPlan')))
                        .withColumn("IsValidEfficacy",when(df_ULLogs.TreatmentTime >= df_applicator.FullEfficacy,lit(1)).otherwise(lit(0)))\
                        .select(df_ULLogs.ShipToAccountID,
                                df_ULLogs.CycleID,
                                df_ULLogs.SoldToAccountID,
                                df_ULLogs.CoolSculptingID,
                                df_AccountRank.PromotionUUID,
                                df_ULLogs.HdrStartDateGeneratedDt,
                                df_ULLogs.HdrStartTimeGeneratedTmstmp,
                                df_ULLogs.CreatedDt,
                                df_InvoiceCycleMonth.InvoiceDate,
                                df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                df_DIMPromotion.PromotionRuleType,
                                df_ULLogs.PlanType,
                                "IsValidEfficacy"))
    
    df_cyclesoverridefnl = getValidCyclesOverride(InvoiceDate)
    df_cycles = (df_cyclesoverridefnl.unionByName(df_ULSource).distinct()
                                     .filter(col("CoolSculptingID") != 'MISSING'))

    windowspec = Window.partitionBy("CycleID").orderBy(asc("FlagRank"))
    windowspecEfficacy = Window.partitionBy("ShipToAccountId","SoldToAccountID","CoolSculptingID","PromotionUUID","HdrStartDateGeneratedDt").orderBy(asc("HdrStartDateGeneratedDt")) 

    df_ullogsfnl = (df_cycles.join(df_accountflagtype,(df_cycles.ShipToAccountID == df_accountflagtype.ShipToAccountId) & (df_cycles.HdrStartDateGeneratedDt.between(df_accountflagtype.FlagStartDate,df_accountflagtype.FlagEndDate)) & (df_cycles.PromotionUUID == df_accountflagtype.PromotionUUID) & (df_cycles.HdrStartDateGeneratedDt.between(df_accountflagtype.Account_EffectiveDate,df_accountflagtype.Account_TerminationDate)),"left")
                .withColumn("rank",row_number().over(windowspec))
                .filter("rank == 1")
                .groupBy(df_cycles.ShipToAccountID.alias("ShipToAccountId"),
                        df_cycles.SoldToAccountID,
                        df_cycles.CoolSculptingID,
                        df_cycles.PromotionUUID,
                        df_cycles.HdrStartDateGeneratedDt,
                        df_cycles.CalendarEndTimeStamp,
                        df_cycles.CalendarStartTimeStamp,
                        df_accountflagtype.FlagType,
                        df_cycles.PromotionRuleType,
                        df_cycles.PlanType,
                        df_cycles.IsValidEfficacy)
                .agg(min(df_cycles.CreatedDt).alias("MinUL_CreatedDt"),
                            max(df_cycles.CreatedDt).alias("MaxUL_CreatedDt"),
                            min(df_cycles.HdrStartTimeGeneratedTmstmp).alias("Min_HdrStartTimeGeneratedTmstmp"),
                            max(df_cycles.InvoiceDate).alias("InvoiceDate"),
                            count("*").alias("TotalCycleCount"))
                .withColumn("IsMultipleEfficacy",size(collect_set(col("IsValidEfficacy")).over(windowspecEfficacy)))
                )
                
    
    #Differentiate based on the US and OUS cycle data 
    df_ullogsconsumer = df_ullogsfnl.filter(col("PromotionRuleType") == 'Consumer')
    df_ullogsconsumershipto = df_ullogsfnl.filter(col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo')
    
    df_ullogsconsumer_fnl = patientClassificationConsumerBased(df_ullogsconsumer,InvoiceDate)
    df_ullogsconsumershipto_fnl = patientClassificationConsumerShipToSoldToBased(df_ullogsconsumershipto,InvoiceDate)

    df_source = df_ullogsconsumer_fnl.unionByName(df_ullogsconsumershipto_fnl).distinct()

    #Already Invoice Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    df_InvoiceDetails = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                            .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                            .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"INNER")\
                                            .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                            .join(df_ConsumerSubscription,["ConsumerSubscriptionUUID","PromotionUUID"],"LEFT")\
                                            .filter(~col("PatientClassificationName").isin('NoPatientAssociationFeeExceptionPostInvoice','NoPatientAssociationFeeExceptionPreInvoice','NoPatientAssociationFeePerCycleException','PerPatientFeeExceptionPostInvoice','PerPatientFeeExceptionPreInvoice','PerPatientFeeException','Exception','NoPatientAssociationFee','PerCycleFee','PerCycleFeeExceptionPostInvoice','PerCycleFeeExceptionPreInvoice'))\
                                            .filter(df_InvoiceAddendumDetails.VersionID == 1)\
                                            .select(df_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                    df_Account.ShipToAccountId,
                                                    df_InvoiceAddendumDetails.CoolSculptingID,
                                                    df_InvoiceAddendumDetails.PromotionUUID,
                                                    df_InvoiceAddendumDetails.CycleDate,
                                                    df_InvoiceAddendumDetails.CycleCount,
                                                    df_PatientClassification.PatientClassificationName,
                                                    df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                                    df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                                    df_InvoiceAddendumDetails.ExceptionMoxieCaseNumber,
                                                    df_InvoiceAddendumDetails.Comments,
                                                    df_InvoiceAddendumDetails.CreatedBy,
                                                    df_InvoiceAddendumDetails.CreatedDate,
                                                    df_ConsumerSubscription.SubscriptionStartDate,
                                                    df_ConsumerSubscription.SubscriptionEndDate,
                                                    df_ConsumerSubscription.SoldToAccountID,
                                                    df_ConsumerSubscription.PilotSubscriptionFlag
                                                    )
                                            
    df_target = df_InvoiceDetails.join(df_invoiceaddendum_invoiceaddendumdetails,(df_InvoiceDetails.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_InvoiceDetails.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_InvoiceDetails.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_InvoiceDetails.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                    .select(df_InvoiceDetails.InvoiceAddendumDetailsUUID,
                            df_InvoiceDetails.ShipToAccountId,
                            df_InvoiceDetails.CoolSculptingID,
                            df_InvoiceDetails.PromotionUUID,
                            df_InvoiceDetails.CycleDate,
                            df_InvoiceDetails.CycleCount,
                            df_InvoiceDetails.PatientClassificationName,
                            df_InvoiceDetails.CalendarStartTimeStamp,
                            df_InvoiceDetails.CalendarEndTimeStamp,
                            df_InvoiceDetails.ExceptionMoxieCaseNumber,
                            df_InvoiceDetails.Comments.alias("Details_Comments"),
                            df_InvoiceDetails.CreatedBy.alias("Details_CreatedBy"),
                            df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced,
                            df_InvoiceDetails.CreatedDate.alias("Details_CreatedDt"),
                            df_InvoiceDetails.PilotSubscriptionFlag).distinct()
                    
    #Add prefix in both ullogs and invoiceaddendumdetails dataframes column names
    df_source = add_Prefix_ColumnName(df_source,"ULLogs_")
    df_target = add_Prefix_ColumnName(df_target,"InvoiceAddendumDetails_")

    df_result = df_source.alias("src").join(df_target.alias("tgt"),(col("src.ULLogs_ShipToAccountId") == col("tgt.InvoiceAddendumDetails_ShipToAccountId") ) & (col("src.ULLogs_CoolSculptingID") == col("tgt.InvoiceAddendumDetails_CoolSculptingID")) & (col("src.ULLogs_HdrStartDateGeneratedDt") == col("tgt.InvoiceAddendumDetails_CycleDate")) & (col("src.ULLogs_PromotionUUID") == col("tgt.InvoiceAddendumDetails_PromotionUUID")),"LEFT")\
                     .withColumn("Comments",when((col("src.ULLogs_IsInvoiced") == 1) | (col("tgt.InvoiceAddendumDetails_IsInvoiced") == 1),"Already Invoice")
                                            .when(((col("src.ULLogs_IsPerPatientFee") == 1) | (col("src.ULLogs_IsNewSubscription") == 1)) & ((~col("src.ULLogs_FlagType").isin('Fraud','OffBoarding_NonP3','TNC','OffBoarding_NonP3_Treatment')) | (col("src.ULLogs_FlagType").isNull())) & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == "PerPatientFee") & (col("src.ULLogs_PlanType") != 'SmallTreatmentPlan'),"PerPatientFee")
                                            .when(((col("src.ULLogs_IsPerPatientFee") != 1) | (col("src.ULLogs_IsDifferentShipTo") == 1) | (col("src.ULLogs_IsSubscriptionInvoiced") == 1)) & (col("src.ULLogs_ConsumerSubscriptionUUID").isNotNull()) & (((~col("src.ULLogs_FlagType").isin('Fraud')) | (col("src.ULLogs_FlagType").isNull())) | (col("src.ULLogs_PlanType") == 'SmallTreatmentPlan'))  & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == "FollowUpVisit"),"FollowUpVisit")
                                            .when(((col("src.ULLogs_IsSubscriptionAdjustedNotInvoiced") == 1) | (col("src.ULLogs_IsSubscriptionAdjusted") == 1) | (col("src.ULLogs_IsSubscriptionAdjustedFollowUp") == 1)),lit("Subscription Adjusted"))
                                            .when(col("tgt.InvoiceAddendumDetails_Details_CreatedBy").like("%AAIOT%"),"DataPatch")
                                            .when((col("src.ULLogs_MaxUL_CreatedDt").between(col("src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("src.ULLogs_CalendarEndTimeStamp"))),"Cycle Received within 20 minutes of Billing Period End Date")
                                            .when((col("tgt.InvoiceAddendumDetails_Details_CreatedDt").between(col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))),"Cycle Received within 20 minutes of Billing Period Start Date")
                                            .when((col("src.ULLogs_FlagType") == 'Fraud') & (col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_PlanType") != 'SmallTreatmentPlan') & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'NonP3PatientFraudViolation'),"Fraud Violation")
                                            .when((col("src.ULLogs_FlagType") == 'TNC') & (col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_ConsumerSubscriptionUUID").isNull()) & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'NonP3Fee'),"TNC Violation")
                                            .when((col("src.ULLogs_FlagType") == 'OffBoarding_NonP3') & (col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_PlanType") != 'SmallTreatmentPlan') & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'NonP3FeeOffboarding'),"NonP3FeeOffboarding")
                                            .when((col("src.ULLogs_FlagType") == 'OffBoarding_NonP3_Treatment') & (col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_PlanType") != 'SmallTreatmentPlan') & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'NonP3FeeOffboardingTreatment'),"NonP3FeeOffboardingTreatment")
                                            .when((col("src.ULLogs_PlanType") == 'SmallTreatmentPlan'),"PerCycleFee")
                                            .when((col("src.ULLogs_IsValidEfficacy") == 0) & (col("src.ULLogs_ConsumerSubscriptionUUID").isNull()) & ((col("src.ULLogs_PlanType") == 'SmallTreatmentPlan')| (col("src.ULLogs_FlagType").isin('OffBoarding_NonP3_Treatment','OffBoarding_NonP3','TNC','Fraud')) | (col("src.ULLogs_CoolSculptingID") == 'MISSING')) & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'PerCycleEfficacyFail'),"PerCycleEfficacyFail")
                                            .when((col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'PerCycleEfficacyFail') & (col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_IsMultipleEfficacy") >1),"Multiple Efficacy on the same day")
                                            .otherwise("Unknown"))
    
    ExceptionCount = df_result.filter(col("Comments")== "Unknown").count()
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comparison Based on Patient Classification:")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
def validateComments(InvoiceDate,TestCaseName):

    #InvoiceException Record
    df_InvoiceException = fetchInvoiceExceptionRecord(InvoiceDate)
    df_InvoiceException = df_InvoiceException.withColumn("IsInvoiceException",lit(1))

    #DifferentShipTo/LateCycle/UpdateCycle/ReturnSubscription
    df_DifferentShipToConsumer = deriveCommentsConsumer(InvoiceDate)
    df_DifferentShipToConsumerShipToSoldTo = deriveCommentsConsumerShipToSoldTo(InvoiceDate)

    df_DifferentShipTo = df_DifferentShipToConsumer.unionByName(df_DifferentShipToConsumerShipToSoldTo)

    #SubscriptionAdjusted
    df_SubscriptionAdjustedConsumer = subscriptionAdjustedConsumer(InvoiceDate)
    df_SubscriptionAdjustedConsumerShipToSoldTo = subscriptionAdjustedConsumerShipToSoldTo(InvoiceDate)

    df_InvoiceAdjusted_fnl = df_SubscriptionAdjustedConsumer.unionByName(df_SubscriptionAdjustedConsumerShipToSoldTo)

    #Already Invoiced Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    df_InvoiceDetailsAccount = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                    .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                    .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"INNER")\
                                    .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                    .join(df_ConsumerSubscription,["ConsumerSubscriptionUUID","PromotionUUID"],"LEFT")\
                                    .filter(df_InvoiceAddendumDetails.VersionID == 1)\
                                    .select(df_Account.ShipToAccountId,
                                            df_InvoiceAddendumDetails.CoolSculptingID,
                                            df_InvoiceAddendumDetails.SoldToAccountID,
                                            df_InvoiceAddendumDetails.PromotionUUID,
                                            df_PatientClassification.PatientClassificationName,
                                            df_InvoiceAddendumDetails.CycleDate,
                                            df_InvoiceAddendumDetails.CycleCount,
                                            df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                            df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                            df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                            df_InvoiceAddendumDetails.CreatedBy,
                                            df_InvoiceAddendumDetails.CreatedDate,
                                            df_InvoiceAddendumDetails.UpdatedDate,
                                            df_InvoiceAddendumDetails.Comments,
                                            df_ConsumerSubscription.PilotSubscriptionFlag
                                            )
                                                
    df_Invoicetarget = df_InvoiceDetailsAccount.join(df_invoiceaddendum_invoiceaddendumdetails,(df_InvoiceDetailsAccount.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_InvoiceDetailsAccount.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_InvoiceDetailsAccount.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_InvoiceDetailsAccount.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                        .select(df_InvoiceDetailsAccount.ShipToAccountId,
                                                df_InvoiceDetailsAccount.CoolSculptingID,
                                                df_InvoiceDetailsAccount.SoldToAccountID,
                                                df_InvoiceDetailsAccount.PromotionUUID,
                                                df_InvoiceDetailsAccount.PatientClassificationName,
                                                df_InvoiceDetailsAccount.CycleDate,
                                                df_InvoiceDetailsAccount.CycleCount,
                                                df_InvoiceDetailsAccount.CalendarStartTimeStamp,
                                                df_InvoiceDetailsAccount.CalendarEndTimeStamp,
                                                df_InvoiceDetailsAccount.IncrementalInvoiceCharge,
                                                df_InvoiceDetailsAccount.CreatedBy,
                                                df_InvoiceDetailsAccount.CreatedDate,
                                                df_InvoiceDetailsAccount.PilotSubscriptionFlag,
                                                df_InvoiceDetailsAccount.Comments.alias("Comments_Detail"),
                                                df_InvoiceDetailsAccount.UpdatedDate,
                                                df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced
                                                ).distinct()
          
    df_result = df_Invoicetarget.alias("AddendumDetails").join(df_DifferentShipTo.alias("ULLogsComment"),(col("AddendumDetails.ShipToAccountId") == col("ULLogsComment.ShipToAccountId")) & (col("AddendumDetails.CoolSculptingID") == col("ULLogsComment.CoolSculptingID")) & (col("AddendumDetails.CycleDate") == col("ULLogsComment.HdrStartDateGeneratedDt")) & (col("AddendumDetails.PromotionUUID") == col("ULLogsComment.PromotionUUID")),"FULL")\
                             .join(df_InvoiceAdjusted_fnl.alias("TruUp"),(col("AddendumDetails.ShipToAccountId") == col("TruUp.ShipToAccountId")) & (col("AddendumDetails.CoolSculptingID") == col("TruUp.CoolSculptingID")) & (col("AddendumDetails.CycleDate") == col("TruUp.HdrStartDateGeneratedDt")) & (col("AddendumDetails.PromotionUUID") == col("TruUp.PromotionUUID")),"FULL")\
                            .join(df_InvoiceException.alias("Exception"),(((col("Exception.ShipToAccountID") == col("AddendumDetails.ShipToAccountId")) | ((col("Exception.ShipToAccountID") != col("AddendumDetails.ShipToAccountId")) & (col("Exception.SoldToAccountID") == col("AddendumDetails.SoldToAccountID")))) & (col("Exception.CoolSculptingID") == col("AddendumDetails.CoolSculptingID")) & (col("AddendumDetails.CycleDate").between(col("Exception.SubscriptionStartDate"),col("Exception.SubscriptionEndDate"))) & (col("Exception.PromotionUUID") == col("AddendumDetails.PromotionUUID")) & (col("Exception.PromotionRuleType") == 'Consumer-ShipTo-SoldTo')) | ((col("Exception.ShipToAccountID") == col("AddendumDetails.ShipToAccountId")) & (col("Exception.CoolSculptingID") == col("AddendumDetails.CoolSculptingID")) & (col("Exception.CycleDate") == col("AddendumDetails.CycleDate")) & (col("Exception.PatientClassificationNameDerived") == col("AddendumDetails.PatientClassificationName")) & (col("Exception.PromotionUUID") == col("AddendumDetails.PromotionUUID"))) | ((col("Exception.CoolSculptingID") == col("AddendumDetails.CoolSculptingID")) & (col("AddendumDetails.CycleDate").between(col("Exception.SubscriptionStartDate"),col("Exception.SubscriptionEndDate"))) & (col("Exception.PromotionUUID") == col("AddendumDetails.PromotionUUID")) & (col("Exception.PromotionRuleType") == 'Consumer')),"FULL")\
                            .withColumn("Comments",when((col("Exception.IsInvoiceException") == '1') & (col("Exception.CommentsDerived") == col("AddendumDetails.Comments_Detail")),"Matched")
                                                .when((col("Exception.IsInvoiceException") == '1') & (col("Exception.CommentsDerived") == "Exception_SubscriptionDateChange") & (col("Exception.Exception_CreatedDt") <= col("AddendumDetails.CreatedDate")),"Record Received after Exception Created")
                                                .when(col("ULLogsComment.IsUpdateCycle") == 1,"Updated Cycle Records will not part of current Invoice")
                                                .when(((trim(col("AddendumDetails.Comments_Detail")) == "") | (col("AddendumDetails.Comments_Detail").isNull())) & (col("ULLogsComment.isDiffernetShipToID").isNull()) & (col("ULLogsComment.IsReturnSubscription").isNull()) & (col("ULLogsComment.IsLateCycle").isNull()) & (col("ULLogsComment.FlagType").isNull()) & (col("Exception.IsInvoiceException").isNull()) & (col("AddendumDetails.PatientClassificationName") != 'PerCycleEfficacyFail') | (col("AddendumDetails.PatientClassificationName").isNull()) ,"Comments are not populated")
                                                .when((col("ULLogsComment.isDiffernetShipToID") == 1) & (col("ULLogsComment.IsLateCycle").isNull()) & (col("AddendumDetails.Comments_Detail") == 'DifferentShipToID_FollowUp'),"Matched")
                                                .when((col("ULLogsComment.isDiffernetShipToID") == 1) & (col("ULLogsComment.IsLateCycle") == 1) & (col("AddendumDetails.Comments_Detail") == 'DifferentShipToID_LateCycle_FollowUp'),"Matched")
                                                .when((col("ULLogsComment.IsReturnSubscription") == 1) & (col("AddendumDetails.Comments_Detail") == 'ReturnSubscription'),"Matched")
                                                .when((col("ULLogsComment.IsLateCycle") == 1) & (col("AddendumDetails.PatientClassificationName") == 'FollowUpVisit') & (col("AddendumDetails.Comments_Detail") == 'LateCycle_FollowUp'),"Matched")
                                                .when((col("ULLogsComment.IsLateCycle") == 1) & (col("AddendumDetails.PatientClassificationName") == 'PerPatientFee') & (col("AddendumDetails.Comments_Detail") == 'LateCycle_NewSubscription'),"Matched")
                                                .when((col("ULLogsComment.IsLateCycle") == 1) & (col("AddendumDetails.PatientClassificationName") == 'NoPatientAssociationFee') & (col("AddendumDetails.Comments_Detail") == 'LateCycle_NoPatientAssociation') & (col("ULLogsComment.IsValidEfficacy") == 1),"LateCycle_NoPatientAssociation")
                                                .when((col("TruUp.IsSubscriptionAdjustedFollowUp") == 1) & (col("AddendumDetails.Comments_Detail") == 'LateCycle_FollowUp_Adjust'),"Matched")
                                                .when((col("TruUp.IsSubscriptionAdjustedFollowUp") == 1) & (col("ULLogsComment.IsReturnSubscription") == '1'),"Return Subscription was tagged in FollowUp Due to TruUp")
                                                .when((col("TruUp.IsSubscriptionAdjusted") == 1) & (col("AddendumDetails.Comments_Detail") == 'LateCycle_SubscriptionAdjust'),"Matched")
                                                .when((col("TruUp.IsSubscriptionAdjustedNotInvoiced") == 1) & (col("ULLogsComment.IsLateCycle") == 1) & (col("AddendumDetails.Comments_Detail") == 'LateCycle_SubscriptionAdjust'),"Matched")
                                                .when((col("ULLogsComment.PreviousPilotSubscriptionFlag")== 'Y') & (col("ULLogsComment.IsReturnSubscription") == 1) ,"Previous subscription was pilot subscription and we don't have any invoice records created previously.")
                                                .when((col("TruUp.IsSubscriptionAdjustedNotInvoiced") == 1) | (col("TruUp.IsSubscriptionReAllignment") == 1) | (col("TruUp.IsSubscriptionAdjusted") == 1),lit("Subscription Adjusted"))
                                                .when((col("ULLogsComment.IsInvoiced") == 1) | (col("AddendumDetails.IsInvoiced") == 1) | (col("Exception.IsInvoiced") == 1),"Already Invoiced")
                                                .when((col("AddendumDetails.CreatedBy").like('AAIOT%')),"Record was added as part of Data Patch")
                                                .when(col("AddendumDetails.CreatedDate").between(col("AddendumDetails.CalendarStartTimeStamp"),col("AddendumDetails.CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES")),"Cycle Received within 20 minutes of Billing Period Start Date")
                                                .when(col("ULLogsComment.UL_CreatedDt").between(col("ULLogsComment.CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("ULLogsComment.CalendarEndTimeStamp")),"Cycle Received within 20 minutes of Billing Period End Date")
                                                .when(col("AddendumDetails.UpdatedDate").between(col("AddendumDetails.CalendarEndTimeStamp"),col("AddendumDetails.CalendarEndTimeStamp")  + expr(f"INTERVAL {BufferTime} MINUTES")),"Additional Cycles received after Billing Period End Date")
                                                .when((col("ULLogsComment.FlagType") == 'Fraud') & (col("ULLogsComment.IsLateCycle").isNull()) & (col("ULLogsComment.PlanType") != 'SmallTreatmentPlan') & (col("AddendumDetails.Comments_Detail") == 'Flag_NonP3PatientFraudViolation') & (col("ULLogsComment.IsValidEfficacy") == 1),"Matched")
                                                .when((col("ULLogsComment.FlagType").isin('Fraud')) & (col("ULLogsComment.IsLateCycle") == 1) & (col("ULLogsComment.PlanType") != 'SmallTreatmentPlan') & (col("AddendumDetails.Comments_Detail") == 'LateCycle_Flag_NonP3PatientFraudViolation') & (col("ULLogsComment.IsValidEfficacy") == 1),"Matched")
                                                .when((col("ULLogsComment.FlagType") == 'TNC') & (col("ULLogsComment.ConsumerSubscriptionUUID_1").isNull()) & (col("AddendumDetails.Comments_Detail") == 'Flag_NonP3Fee') & (col("ULLogsComment.IsValidEfficacy") == 1),"Matched")
                                                .when((col("ULLogsComment.FlagType").isin('TNC','OffBoarding_NonP3_Treatment','OffBoarding_NonP3')) & (col("ULLogsComment.ConsumerSubscriptionUUID_1").isNotNull()) & (col("AddendumDetails.PatientClassificationName") == 'FollowUpVisit') & ((trim(col("AddendumDetails.Comments_Detail")) == "") | (col("AddendumDetails.Comments_Detail").isNull())),"Matched")
                                                .when(((col("ULLogsComment.FlagType").isin('TNC','Fraud','OffBoarding_NonP3_Treatment','OffBoarding_NonP3')) | (col("ULLogsComment.PlanType") == 'SmallTreatmentPlan')) & (col("ULLogsComment.CoolSculptingID") == 'MISSING') & (col("ULLogsComment.IsValidEfficacy") == 1),"No Patient Association Fee")
                                                .when(((col("ULLogsComment.FlagType").isin('TNC','Fraud','OffBoarding_NonP3_Treatment','OffBoarding_NonP3')) & (col("ULLogsComment.PlanType") == 'SmallTreatmentPlan')) & (col("ULLogsComment.IsValidEfficacy") == 1),"Per Cycle Fee")
                                                .when((col("ULLogsComment.FlagType").isin('OffBoarding_NonP3_Treatment','OffBoarding_NonP3')) & (col("ULLogsComment.ConsumerSubscriptionUUID_1").isNull()) & (col("ULLogsComment.PlanType") != 'SmallTreatmentPlan') & (col("AddendumDetails.Comments_Detail") == 'OffBoardingFlag_NonP3Fee') & (col("ULLogsComment.IsValidEfficacy") == 1) ,"Matched")
                                                .when((col("ULLogsComment.IsLateCycle") == 1) & (col("ULLogsComment.PlanType") == 'SmallTreatmentPlan') & (col("AddendumDetails.PatientClassificationName") == 'PerCycleFee') & (col("AddendumDetails.Comments_Detail") == 'LateCycle_PerCycleFee') & (col("ULLogsComment.IsValidEfficacy") == 1),"Matched")
                                                .when((col("ULLogsComment.IsValidEfficacy") == 0) & (((col("ULLogsComment.PlanType") == 'SmallTreatmentPlan') & (col("ULLogsComment.ConsumerSubscriptionUUID_1").isNull()))| (col("ULLogsComment.FlagType").isin('OffBoarding_NonP3_Treatment','OffBoarding_NonP3','TNC','Fraud')) | (col("ULLogsComment.CoolSculptingID") == 'MISSING')) & (col("AddendumDetails.PatientClassificationName") == 'PerCycleEfficacyFail') & (col("AddendumDetails.Comments_Detail") == 'PerCycleEfficacyFail'),"PerCycleEfficacyFail")
                                                .when((col("ULLogsComment.IsMultipleEfficacy") > 1),"Multiple Efficacy changes on the same day")
                                                .when((col("ULLogsComment.IsMultiplePlanType") > 1) & (col("ULLogsComment.FlagType").isin('TNC','Fraud','OffBoarding_NonP3_Treatment','OffBoarding_NonP3')) & (col("ULLogsComment.PlanType") != 'SmallTreatmentPlan'),"Multiple PlanType changes on the same day")
                                                .otherwise("Unknown"))\
                        .select(col("AddendumDetails.ShipToAccountId").alias("AddendumDetails_ShipToAccountId"),
                                    col("AddendumDetails.CoolSculptingID").alias("AddendumDetails_CoolSculptingID"),
                                    col("AddendumDetails.PromotionUUID").alias("AddendumDetails_PromotionUUID"),
                                    col("AddendumDetails.CycleDate").alias("AddendumDetails_CycleDate"),
                                    col("AddendumDetails.CycleCount").alias("AddendumDetails_CycleCount"),
                                    col("AddendumDetails.Comments_Detail").alias("AddendumDetails_Comments"),
                                    col("AddendumDetails.CreatedBy").alias("AddendumDetails_CreatedBy"),
                                    col("AddendumDetails.CreatedDate").alias("AddendumDetails_CreatedDate"),
                                    col("AddendumDetails.PilotSubscriptionFlag").alias("AddendumDetails_PilotSubscriptionFlag"),
                                    col("AddendumDetails.PatientClassificationName").alias("PatientClassificationName"),
                                    col("AddendumDetails.IsInvoiced").alias("AddendumDetails_IsInvoicedDerived"),
                                    coalesce(col("ULLogsComment.ShipToAccountId"),col("TruUp.ShipToAccountId"),col("Exception.ShipToAccountID")).alias("ULLogs_ShipToAccountID"),
                                    coalesce(col("ULLogsComment.CoolSculptingID"),col("TruUp.CoolSculptingID"),col("Exception.CoolSculptingID")).alias("ULLogs_CoolSculptingID"),
                                    coalesce(col("ULLogsComment.HdrStartDateGeneratedDt"),col("TruUp.HdrStartDateGeneratedDt"),col("Exception.CycleDate")).alias("ULLogs_CycleDate"),
                                    col("ULLogsComment.ConsumerSubscriptionUUID_1").alias("ULLogsComment_ConsumerSubscriptionUUID"),
                                    col("ULLogsComment.IsInvoiced").alias("ULLogsComment_IsInvoiced"),
                                    col("ULLogsComment.isDiffernetShipToID").alias("ULLogsComment_isDiffernetShipToID"),
                                    col("ULLogsComment.IsPerPatientFee").alias("ULLogsComment_IsPerPatientFee"),
                                    col("ULLogsComment.IsReturnSubscription").alias("ULLogsComment_IsReturnSubscription"),
                                    col("ULLogsComment.IsUpdateCycle").alias("ULLogsComment_IsUpdateCycle"),
                                    col("ULLogsComment.IsLateCycle").alias("ULLogsComment_IsLateCycle"),
                                    col("ULLogsComment.FlagType").alias("ULLogs_FlagType"),
                                    col("ULLogsComment.PreviousPilotSubscriptionFlag").alias("ULLogs_PreviousPilotSubscriptionFlag"),
                                    col("ULLogsComment.PlanType").alias("ULLogs_PlanType"),
                                    col("ULLogsComment.IsValidEfficacy").alias("ULLogs_IsValidEfficacy"),
                                    col("ULLogsComment.IsMultipleEfficacy").alias("ULLogs_IsMultipleEfficacy"),
                                    col("ULLogsComment.IsMultiplePlanType").alias("ULLogs_IsMultiplePlanType"),
                                    col("TruUp.IsSubscriptionAdjustedFollowUp").alias("IsSubscriptionAdjustedFollowUpDerived"),
                                    col("TruUp.IsSubscriptionAdjusted").alias("IsSubscriptionAdjustedDerived"),
                                    col("TruUp.IsSubscriptionReAllignment").alias("IsSubscriptionReAllignmentDerived"),
                                    col("TruUp.IsSubscriptionAdjustedNotInvoiced").alias("IsSubscriptionAdjustedNotInvoicedDerived"),
                                    col("Exception.MoxieCaseNumber").alias("Exception_MoxieCaseNumber"),
                                    col("Exception.IsInvoiceException").alias("Exception_IsInvoiceException"),
                                    col("Exception.CommentsDerived").alias("Exception_CommentsDerived"),
                                    "Comments") 
                            
    ExceptionCount = df_result.filter(col("Comments")== "Unknown").count() 
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comments Validation:")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
def validateNoPatientAssociationFee(InvoiceDate,TestCaseName):

    #Valid P3 Eligible Cycles and calculate the Invoice Amount based on cycles
    df_ULSource = validP3EligibleCycles(InvoiceDate)
    df_ULSource= df_ULSource.filter(col("CoolSculptingID") == 'MISSING')\
                            .withColumn("IncrementalInvoiceCharge",col("TotalCycleCount")*260)
    
    #Read No Patient Association Fee records from InvoiceAddendumDetails
    df_InvoiceAddendumDetails_shipto = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                                    .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                                    .filter("VersionID == 1")\
                                                                    .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                                                    .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(col("CalendarStartTimeStamp"), col("CalendarEndTimeStamp"))) & (col("InvoiceDate") == lit(f'{InvoiceDate}')),"INNER")\
                                                                    .filter(col("PatientClassificationName").isin('NoPatientAssociationFee'))\
                                                                    .select(df_Account.ShipToAccountId,
                                                                            df_InvoiceAddendumDetails.CoolSculptingID,
                                                                            df_InvoiceAddendumDetails.PromotionUUID,
                                                                            df_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                                            df_PatientClassification.PatientClassificationName,
                                                                            df_InvoiceAddendumDetails.CycleCount,
                                                                            df_InvoiceAddendumDetails.VersionID,
                                                                            df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                                            df_InvoiceAddendumDetails.CreatedDate.alias("Details_CreatedDate"),            
                                                                            df_InvoiceAddendumDetails.CycleDate,
                                                                            df_InvoiceAddendumDetails.Comments.alias("Details_comments"),
                                                                            df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                                                            df_InvoiceCycleMonth.CalendarEndTimeStamp)
    
    #Aggregate records for comparing with Ullogs
    df_InvoiceAddendumDetails_shiptoAgg = df_InvoiceAddendumDetails_shipto.groupBy("ShipToAccountId",
                                                                                "CoolSculptingID",
                                                                                "PromotionUUID",
                                                                                "PatientClassificationName",
                                                                                "CycleDate",
                                                                                "CalendarStartTimeStamp",
                                                                                "CalendarEndTimeStamp")\
                                                                        .agg(sum(col("IncrementalInvoiceCharge")).alias("IncrementalInvoiceCharge"),
                                                                                sum(col("CycleCount")).alias("CycleCount"),
                                                                                max(col("Details_CreatedDate")).alias("Details_CreatedDate"))

    df_ULSource = add_Prefix_ColumnName(df_ULSource,"ULLogs_")
    df_InvoiceAddendumDetails_shiptoAgg = add_Prefix_ColumnName(df_InvoiceAddendumDetails_shiptoAgg,"InvoiceAddendumDetails_") 

    df_result = df_ULSource.alias("src").join(df_InvoiceAddendumDetails_shiptoAgg .alias("tgt"),((col("src.ULLogs_ShipToAccountId") == col("tgt.InvoiceAddendumDetails_ShipToAccountId")) & (col("src.ULLogs_CoolSculptingID") == col("tgt.InvoiceAddendumDetails_CoolSculptingID"))  & (col("src.ULLogs_HdrStartDateGeneratedDt") == col("tgt.InvoiceAddendumDetails_CycleDate")) & (col("src.ULLogs_PromotionUUID") == col("tgt.InvoiceAddendumDetails_PromotionUUID"))),"FULL")\
                                .withColumn("Comments",when((col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_IncrementalInvoiceCharge") == col("tgt.InvoiceAddendumDetails_IncrementalInvoiceCharge")) & (col("src.ULLogs_TotalCycleCount") == col("tgt.InvoiceAddendumDetails_CycleCount")),"Matched")
                                                        .when((col("src.ULLogs_UL_CreatedDt").between(col("src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("src.ULLogs_CalendarEndTimeStamp"))),"Cycle Received within 20 minutes of Billing Period End Date")
                                                        .when((col("tgt.InvoiceAddendumDetails_Details_CreatedDate").between(col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))),"Cycle Received within 20 minutes of Billing Period Start Date")
                                                        .when((col("src.ULLogs_IsValidEfficacy") == 0),"PerCycleEfficacyFail")
                                                        .otherwise("Unknown"))
    
    ExceptionCount = df_result.filter(col("Comments")== "Unknown").count() 
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"NoPatientAssociationFee Validation:")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
#Verify 950 dollars are charged and Consumber subscription End Date was calculated corrrectly.
def validatePerPatientFee(InvoiceDate,TestCaseName):

    #P3 Eligible Cycles
    df_ULSource = validP3EligibleCycles(InvoiceDate)

    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Differentiate based on the US and OUS cycle data 
    df_ullogsconsumer = df_ULSource.filter(col("PromotionRuleType") == 'Consumer')
    df_ullogsconsumershipto = df_ULSource.filter(col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo')

    df_ullogsconsumer_fnl = perPatientFeeValidationConsumerBased(df_ullogsconsumer,InvoiceDate)
    df_ullogsconsumershipto_fnl = perPatientFeeValidationConsumerShipToSoldToBased(df_ullogsconsumershipto,InvoiceDate)

    df_SourceClassification = df_ullogsconsumer_fnl.unionByName(df_ullogsconsumershipto_fnl).distinct()
                
    df_InvoiceDetailsAccount = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                    .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                    .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(df_InvoiceCycleMonth.CalendarStartTimeStamp, df_InvoiceCycleMonth.CalendarEndTimeStamp)) & (df_InvoiceCycleMonth.InvoiceDate == lit(f'{InvoiceDate}')),"INNER")\
                                    .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                    .join(df_ConsumerSubscription,["ConsumerSubscriptionUUID","PromotionUUID"],"LEFT")\
                                    .filter(df_InvoiceAddendumDetails.VersionID == 1)\
                                    .select(df_Account.ShipToAccountId,
                                            df_InvoiceAddendumDetails.CoolSculptingID,
                                            df_InvoiceAddendumDetails.SoldToAccountID,
                                            df_InvoiceAddendumDetails.PromotionUUID,
                                            df_PatientClassification.PatientClassificationName,
                                            df_InvoiceAddendumDetails.CycleDate,
                                            df_InvoiceAddendumDetails.CycleCount,
                                            df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                            df_InvoiceCycleMonth.CalendarEndTimeStamp,
                                            df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                            df_InvoiceAddendumDetails.CreatedBy,
                                            df_InvoiceAddendumDetails.CreatedDate,
                                            df_ConsumerSubscription.PilotSubscriptionFlag,
                                            df_ConsumerSubscription.SubscriptionStartDate,
                                            df_ConsumerSubscription.SubscriptionEndDate.alias("ConsumerSubscriptionEndDate")
                                            )
                                                
    df_Invoicetarget = df_InvoiceDetailsAccount.join(df_invoiceaddendum_invoiceaddendumdetails,(df_InvoiceDetailsAccount.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_InvoiceDetailsAccount.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_InvoiceDetailsAccount.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_InvoiceDetailsAccount.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                    .select(df_InvoiceDetailsAccount.ShipToAccountId,
                                            df_InvoiceDetailsAccount.CoolSculptingID,
                                            df_InvoiceDetailsAccount.SoldToAccountID,
                                            df_InvoiceDetailsAccount.PromotionUUID,
                                            df_InvoiceDetailsAccount.PatientClassificationName,
                                            df_InvoiceDetailsAccount.CycleDate,
                                            df_InvoiceDetailsAccount.CycleCount,
                                            df_InvoiceDetailsAccount.CalendarStartTimeStamp,
                                            df_InvoiceDetailsAccount.CalendarEndTimeStamp,
                                            df_InvoiceDetailsAccount.IncrementalInvoiceCharge,
                                            df_InvoiceDetailsAccount.CreatedBy,
                                            df_InvoiceDetailsAccount.CreatedDate,
                                            df_InvoiceDetailsAccount.PilotSubscriptionFlag,
                                            df_InvoiceDetailsAccount.SubscriptionStartDate,
                                            df_InvoiceDetailsAccount.ConsumerSubscriptionEndDate,
                                            df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced
                                            )\
                                    .filter(df_InvoiceDetailsAccount.PatientClassificationName.isin('PerPatientFee'))
    
    #Add prefix in both ullogs and invoiceaddendumdetails dataframes column names
    df_SourceClassification = add_Prefix_ColumnName(df_SourceClassification,"ULLogs_")
    df_Invoicetarget = add_Prefix_ColumnName(df_Invoicetarget,"InvoiceAddendumDetails_") 

    df_result = df_SourceClassification.alias("src").join(df_Invoicetarget.alias("tgt"),(col("src.ULLogs_ShipToAccountId") == col("tgt.InvoiceAddendumDetails_ShipToAccountId")) & (col("src.ULLogs_CoolSculptingID") == col("tgt.InvoiceAddendumDetails_CoolSculptingID"))  & (col("src.ULLogs_HdrStartDateGeneratedDt") == col("tgt.InvoiceAddendumDetails_CycleDate")) & (col("src.ULLogs_PromotionUUID") == col("tgt.InvoiceAddendumDetails_PromotionUUID")),"FULL")\
                                        .withColumn("Comments",when((col("src.ULLogs_isPerPatinetFee").isNull()) & (col("tgt.InvoiceAddendumDetails_PatientClassificationName").isNull()),lit("Not a Per Patient Fee Record"))
                                                .when((col("src.ULLogs_isPerPatinetFee") == 1) & (col("tgt.InvoiceAddendumDetails_IncrementalInvoiceCharge") == 950) & (col("src.ULLogs_SubscriptionEndDateDrived") == col("tgt.InvoiceAddendumDetails_ConsumerSubscriptionEndDate")) ,lit("Matched"))
                                                .when((col("tgt.InvoiceAddendumDetails_IsInvoiced") == 1) | (col("src.ULLogs_IsInvoiced_ULLogs") == 1),lit("Already Invoiced"))
                                                .when((col("src.ULLogs_isPerPatinetFee") == 1) & (col("tgt.InvoiceAddendumDetails_IncrementalInvoiceCharge") == 950) & (col("src.ULLogs_IsException") == 1),lit("Exception Record"))
                                                .when((col("src.ULLogs_isPerPatinetFee") == 1) & (col("src.ULLogs_IsSubscriptionAdjustedFollowUp") == 1),lit("Subscription Adjusted FollowUp"))
                                                .when((col("src.ULLogs_IsSubscriptionAdjustedNotInvoiced") == 1),lit("Subscription Adjusted"))
                                                .when((col("tgt.InvoiceAddendumDetails_CreatedBy").like('AAIOT%')),"Record was added as part of Data Patch")
                                                .when(col("tgt.InvoiceAddendumDetails_CreatedDate").between(col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES")),"Cycle Received within 20 minutes of Billing Period Start Date")
                                                .when(col("src.ULLogs_UL_CreatedDt").between(col("src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("src.ULLogs_CalendarEndTimeStamp")),"Cycle Received within 20 minutes of Billing Period End Date")
                                                .when(col("src.ULLogs_FlagType").isin('TNC','Fraud','OffBoarding_NonP3','OffBoarding_NonP3_Treatment'),"Offboarding Violation")
                                                .otherwise("Unknown"))
                                  
    ExceptionCount = df_result.filter("Comments == 'Unknown'").count()   
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comparison between ULLogs and FactInvoiceAddendumDetails for Per Patient Fee Calculation")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
#Verify 200 dollars are charged corrrectly for SmallTreatmentPlan.
def validatePerCycleFee(InvoiceDate,TestCaseName):

    #P3 Eligible Cycles
    df_ULSource = validP3EligibleCycles(InvoiceDate)

    #Already Invoiced Records
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()

    #Differentiate based on the US and OUS cycle data  
    df_ullogsconsumer = df_ULSource.filter((col("PromotionRuleType") == 'Consumer') & (col("PlanType") == 'SmallTreatmentPlan'))
    df_ullogsconsumershipto = df_ULSource.filter((col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo') & (col("PlanType") == 'SmallTreatmentPlan'))

    df_ullogsconsumer_fnl = perCycleFeeValidationConsumerBased(df_ullogsconsumer,InvoiceDate)
    df_ullogsconsumershipto_fnl = perCycleFeeValidationConsumerShipToSoldToBased(df_ullogsconsumershipto,InvoiceDate)

    df_SourceClassification = df_ullogsconsumer_fnl.unionByName(df_ullogsconsumershipto_fnl).distinct()
                
    #Read Per Cycle Fee records from InvoiceAddendumDetails
    df_InvoiceAddendumDetails_shipto = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                                    .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                                    .filter("VersionID == 1")\
                                                                    .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                                                    .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(col("CalendarStartTimeStamp"), col("CalendarEndTimeStamp"))) & (col("InvoiceDate") == lit(f'{InvoiceDate}')),"INNER")\
                                                                    .filter((df_PatientClassification.PatientClassificationName).isin('PerCycleFee'))\
                                                                    .select(df_Account.ShipToAccountId,
                                                                            df_InvoiceAddendumDetails.CoolSculptingID,
                                                                            df_InvoiceAddendumDetails.PromotionUUID,
                                                                            df_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                                            df_PatientClassification.PatientClassificationName,
                                                                            df_InvoiceAddendumDetails.CycleCount,
                                                                            df_InvoiceAddendumDetails.VersionID,
                                                                            df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                                            df_InvoiceAddendumDetails.CreatedDate.alias("Details_CreatedDate"),            
                                                                            df_InvoiceAddendumDetails.CycleDate,
                                                                            df_InvoiceAddendumDetails.Comments.alias("Details_comments"),
                                                                            df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                                                            df_InvoiceCycleMonth.CalendarEndTimeStamp)
    
    #Aggregate records for comparing with Ullogs
    df_Invoicetarget = df_InvoiceAddendumDetails_shipto.join(df_invoiceaddendum_invoiceaddendumdetails,(df_InvoiceAddendumDetails_shipto.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId) & (df_InvoiceAddendumDetails_shipto.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_InvoiceAddendumDetails_shipto.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_InvoiceAddendumDetails_shipto.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
                                                        .select(df_InvoiceAddendumDetails_shipto.ShipToAccountId,
                                                                df_InvoiceAddendumDetails_shipto.CoolSculptingID,
                                                                df_InvoiceAddendumDetails_shipto.PromotionUUID,
                                                                df_InvoiceAddendumDetails_shipto.InvoiceAddendumDetailsUUID,
                                                                df_InvoiceAddendumDetails_shipto.PatientClassificationName,
                                                                df_InvoiceAddendumDetails_shipto.CycleCount,
                                                                df_InvoiceAddendumDetails_shipto.VersionID,
                                                                df_InvoiceAddendumDetails_shipto.IncrementalInvoiceCharge,
                                                                df_InvoiceAddendumDetails_shipto.Details_CreatedDate,            
                                                                df_InvoiceAddendumDetails_shipto.CycleDate,
                                                                df_InvoiceAddendumDetails_shipto.Details_comments,
                                                                df_InvoiceAddendumDetails_shipto.CalendarStartTimeStamp,
                                                                df_InvoiceAddendumDetails_shipto.CalendarEndTimeStamp,
                                                                df_invoiceaddendum_invoiceaddendumdetails.IsInvoiced)\
                                                        .groupBy("ShipToAccountId",
                                                                "CoolSculptingID",
                                                                "PromotionUUID",
                                                                "PatientClassificationName",
                                                                "CycleDate",
                                                                "CalendarStartTimeStamp",
                                                                "CalendarEndTimeStamp",
                                                                "IsInvoiced")\
                                                        .agg(sum(col("IncrementalInvoiceCharge")).alias("IncrementalInvoiceCharge"),
                                                                sum(col("CycleCount")).alias("CycleCount"),
                                                                max(col("Details_CreatedDate")).alias("Details_CreatedDate"))
    
    #Add prefix in both ullogs and invoiceaddendumdetails dataframes column names
    df_SourceClassification = add_Prefix_ColumnName(df_SourceClassification,"ULLogs_")
    df_Invoicetarget = add_Prefix_ColumnName(df_Invoicetarget,"InvoiceAddendumDetails_") 

    df_result = df_SourceClassification.alias("src").join(df_Invoicetarget.alias("tgt"),(col("src.ULLogs_ShipToAccountId") == col("tgt.InvoiceAddendumDetails_ShipToAccountId")) & (col("src.ULLogs_CoolSculptingID") == col("tgt.InvoiceAddendumDetails_CoolSculptingID"))  & (col("src.ULLogs_HdrStartDateGeneratedDt") == col("tgt.InvoiceAddendumDetails_CycleDate")) & (col("src.ULLogs_PromotionUUID") == col("tgt.InvoiceAddendumDetails_PromotionUUID")),"FULL")\
                                        .withColumn("Comments",when((col("src.ULLogs_ConsumerSubscriptionUUID").isNull()) & (col("src.ULLogs_IsValidEfficacy") == 1)  & (col("src.ULLogs_InvoiceAmountDerived") == col("tgt.InvoiceAddendumDetails_IncrementalInvoiceCharge")),lit("Matched"))
                                                .when((col("tgt.InvoiceAddendumDetails_IsInvoiced") == 1) | (col("src.ULLogs_IsInvoiced_ULLogs") == 1),"Already Invoiced")
                                                .when((col("src.ULLogs_IsException") == 1),lit("Exception Record")) 
                                                .when((col("src.ULLogs_ConsumerSubscriptionUUID").isNotNull()),lit("Subscription was already there"))
                                                .when((col("src.ULLogs_CoolSculptingID") == 'MISSING'),lit("NoPatientAssociationFee"))
                                                .when(col("tgt.InvoiceAddendumDetails_Details_CreatedDate").between(col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES")),"Cycle Received within 20 minutes of Billing Period Start Date")
                                                .when(col("src.ULLogs_UL_CreatedDt").between(col("src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("src.ULLogs_CalendarEndTimeStamp")),"Cycle Received within 20 minutes of Billing Period End Date")
                                                .when((col("src.ULLogs_IsValidEfficacy") == 0) & (col("src.ULLogs_ConsumerSubscriptionUUID").isNull()),"Invalid Efficacy Time")  
                                                .otherwise("Unknown"))
                                  
    ExceptionCount = df_result.filter("Comments == 'Unknown'").count()   
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comparison between ULLogs and FactInvoiceAddendumDetails for Per Patient Fee Calculation")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
def validateOffboardingInvoiceAmount(InvoiceDate,TestCaseName):

    #Valid P3 Cycles
    df_P3ULLogs = validP3EligibleCycles(InvoiceDate)

    #Differentiate based on the US and OUS cycle data 
    df_ullogsconsumer = df_P3ULLogs.filter(col("PromotionRuleType") == 'Consumer')
    df_ullogsconsumershipto = df_P3ULLogs.filter(col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo')

    df_ullogsconsumer_fnl = offboardingrulesConsumerBased(df_ullogsconsumer,InvoiceDate)
    df_ullogsconsumershipto_fnl = offboardingrulesConsumerShipToSoldToBased(df_ullogsconsumershipto,InvoiceDate)

    df_P3Cycle_Subscription = df_ullogsconsumer_fnl.unionByName(df_ullogsconsumershipto_fnl).distinct()

    #Derive Invoice Amount
    df_P3CycleInvoiceAmt = df_P3Cycle_Subscription.withColumn("InvoiceAmount_Derived",when(df_P3Cycle_Subscription.FlagType == 'Fraud',df_P3Cycle_Subscription.ListPrice * df_P3Cycle_Subscription.TotalCycleCount)
                                    .when((df_P3Cycle_Subscription.FlagType.isin('TNC','OffBoarding_NonP3','OffBoarding_NonP3_Treatment')) & (df_P3Cycle_Subscription.IsFollowUp == 1) & (df_P3Cycle_Subscription.IsActiveSubscription == 1),0)
                                    .when((df_P3Cycle_Subscription.FlagType == 'TNC') & (df_P3Cycle_Subscription.IsActiveSubscription == 0),df_P3Cycle_Subscription.ListPrice * df_P3Cycle_Subscription.TotalCycleCount)
                                    .when(df_P3Cycle_Subscription.FlagType == 'OffBoarding_NonP3',df_P3Cycle_Subscription.ListPrice * df_P3Cycle_Subscription.TotalCycleCount)
                                    .when(df_P3Cycle_Subscription.FlagType == 'OffBoarding_NonP3_Treatment',df_P3Cycle_Subscription.ListPrice * df_P3Cycle_Subscription.TotalCycleCount)
                                    )

    #Current Month record from FactInvoiceAddendumDetails
    df_Invoicetarget = currentInvoiceRecord(InvoiceDate)

    #Already Invoiced Record
    df_invoiceaddendum_invoiceaddendumdetails = previousInvoicedRecord()
                                    
    #Filter out PerCycleFee and PerCycleEfficacyFlag records for validation
    df_Invoicetarget = df_Invoicetarget.filter(~col("PatientClassificationName").isin('PerCycleEfficacyFail','PerCycleFee'))

    #Join with the fact_invoiceaddendum_invoiceaddendumdetails table to check the cycle information was already processed(Invoice Details could be processed in previous billing period and later exception was created)
    df_Invoicetarget = df_Invoicetarget.alias("InvoiceAddendum").join(df_invoiceaddendum_invoiceaddendumdetails.alias("Invoiced"),(df_Invoicetarget.ShipToAccountId == df_invoiceaddendum_invoiceaddendumdetails.ShipToAccountId ) & (df_Invoicetarget.CoolSculptingID == df_invoiceaddendum_invoiceaddendumdetails.CoolSculptingID) & (df_Invoicetarget.CycleDate == df_invoiceaddendum_invoiceaddendumdetails.CycleDate) & (df_Invoicetarget.PromotionUUID == df_invoiceaddendum_invoiceaddendumdetails.PromotionUUID),"LEFT")\
            .select(col("InvoiceAddendum.ShipToAccountId"),
                    col("InvoiceAddendum.CoolSculptingID"),
                    col("InvoiceAddendum.PromotionUUID"),
                    col("InvoiceAddendum.CycleDate"),
                    col("InvoiceAddendum.CycleCount"),
                    col("InvoiceAddendum.PatientClassificationName"),
                    col("InvoiceAddendum.IncrementalInvoiceCharge"),
                    col("InvoiceAddendum.CalendarStartTimeStamp"),
                    col("InvoiceAddendum.CalendarEndTimeStamp"),
                    col("InvoiceAddendum.CreatedBy").alias("Details_CreatedBy"),
                    col("InvoiceAddendum.Details_UpdatedDt"),
                    col("Invoiced.IsInvoiced"),
                    "Details_CreatedDt",
                    "PilotSubscriptionFlag").distinct()

    #Add prefix in both ullogs and invoiceaddendumdetails dataframes column names
    df_P3CycleInvoiceAmt = add_Prefix_ColumnName(df_P3CycleInvoiceAmt,"ULLogs_")
    df_Invoicetarget = add_Prefix_ColumnName(df_Invoicetarget,"InvoiceAddendumDetails_") 


    df_result = df_P3CycleInvoiceAmt.alias("src").join(df_Invoicetarget.alias("tgt"),(col("src.ULLogs_ShipToAccountId") == col("tgt.InvoiceAddendumDetails_ShipToAccountId")) & (col("src.ULLogs_PromotionUUID") == col("tgt.InvoiceAddendumDetails_PromotionUUID")) & (col("src.ULLogs_HdrStartDateGeneratedDt") == col("tgt.InvoiceAddendumDetails_CycleDate")) & (col("src.ULLogs_CoolSculptingID") == col("tgt.InvoiceAddendumDetails_CoolSculptingID")),"LEFT")\
                                    .withColumn("Comments",when((col("src.ULLogs_IsValidEfficacy") == 1) & (col("src.ULLogs_InvoiceAmount_Derived") == col("tgt.InvoiceAddendumDetails_IncrementalInvoiceCharge")), "Invoice Amount Matched")
                                                    .when((col("src.ULLogs_UL_CreatedDt").between(col("src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("src.ULLogs_CalendarEndTimeStamp"))),"Cycle Received within 20 minutes of Billing Period End Date")
                                                    .when((col("tgt.InvoiceAddendumDetails_Details_CreatedDt").between(col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))),"Cycle Received within 20 minutes of Billing Period Start Date")
                                                    .when(col("src.ULLogs_PlanType") == 'SmallTreatmentPlan',"SmallTreatmentPlan")
                                                    .when((col("src.ULLogs_IsValidEfficacy") == 0) & (col("src.ULLogs_FlagType").isin('Fraud','TNC','OffBoarding_NonP3','OffBoarding_NonP3_Treatment')),"Invalid Efficacy Time")
                                                    .otherwise("Unknown")
                                                    )

    ExceptionCount = df_result.filter("Comments == 'Unknown'").count()   
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"Comparison between ULLogs and FactInvoiceAddendumDetails for Offboarding Invoice Amount")
    df_result.filter("Comments == 'Unknown'").display()

In [0]:
# validate if invoice amount was charged for the cycles that doesn't meet Efficacy time. 
def validatePerCycleEfficacyFailFee(InvoiceDate,TestCaseName):

    #Valid P3 Eligible Cycles with Efficacy time was not matched
    df_ULSource = validP3EligibleCycles(InvoiceDate)
    df_ULSource= df_ULSource.filter(col("IsValidEfficacy") == 0)

    #Differentiate based on the US and OUS cycle data 
    df_ullogsconsumer = df_ULSource.filter(col("PromotionRuleType") == 'Consumer')
    # df_ullogsconsumershipto = df_ULSource.filter(col("PromotionRuleType") == 'Consumer-ShipTo-SoldTo')

    df_SourceClassification = perCycleEfficacyFailValidationConsumerBased(df_ullogsconsumer,InvoiceDate)
    
    #Read PreCycleEfficacyFail records from InvoiceAddendumDetails
    df_InvoiceAddendumDetails_shipto = df_InvoiceAddendumDetails.join(df_DIMPromotion,["PromotionUUID"],"INNER")\
                                                                    .join(df_Account,["ShipToAccountUUID","PromotionUUID"],"INNER")\
                                                                    .filter("VersionID == 1")\
                                                                    .join(df_PatientClassification,["PatientClassificationUUID","PromotionUUID"],"INNER")\
                                                                    .join(df_InvoiceCycleMonth,(df_InvoiceAddendumDetails.CreatedDate.between(col("CalendarStartTimeStamp"), col("CalendarEndTimeStamp"))) & (col("InvoiceDate") == lit(f'{InvoiceDate}')),"INNER")\
                                                                    .filter(col("PatientClassificationName").isin('PerCycleEfficacyFail'))\
                                                                    .select(df_Account.ShipToAccountId,
                                                                            df_InvoiceAddendumDetails.CoolSculptingID,
                                                                            df_InvoiceAddendumDetails.PromotionUUID,
                                                                            df_InvoiceAddendumDetails.InvoiceAddendumDetailsUUID,
                                                                            df_PatientClassification.PatientClassificationName,
                                                                            df_InvoiceAddendumDetails.CycleCount,
                                                                            df_InvoiceAddendumDetails.VersionID,
                                                                            df_InvoiceAddendumDetails.IncrementalInvoiceCharge,
                                                                            df_InvoiceAddendumDetails.CreatedDate.alias("Details_CreatedDate"),            
                                                                            df_InvoiceAddendumDetails.CycleDate,
                                                                            df_InvoiceAddendumDetails.Comments.alias("Details_comments"),
                                                                            df_InvoiceCycleMonth.CalendarStartTimeStamp,
                                                                            df_InvoiceCycleMonth.CalendarEndTimeStamp)
    
    #Aggregate records for comparing with Ullogs
    df_InvoiceAddendumDetails_shiptoAgg = df_InvoiceAddendumDetails_shipto.groupBy("ShipToAccountId",
                                                                                "CoolSculptingID",
                                                                                "PromotionUUID",
                                                                                "PatientClassificationName",
                                                                                "CycleDate",
                                                                                "CalendarStartTimeStamp",
                                                                                "CalendarEndTimeStamp")\
                                                                        .agg(sum(col("IncrementalInvoiceCharge")).alias("IncrementalInvoiceCharge"),
                                                                                sum(col("CycleCount")).alias("CycleCount"),
                                                                                max(col("Details_CreatedDate")).alias("Details_CreatedDate"))

    # Adding Prefix to the column names to identify which field was derived from which dataframe
    df_SourceClassification = add_Prefix_ColumnName(df_SourceClassification,"ULLogs_")
    df_InvoiceAddendumDetails_shiptoAgg = add_Prefix_ColumnName(df_InvoiceAddendumDetails_shiptoAgg,"InvoiceAddendumDetails_") 

    df_result = df_SourceClassification.alias("src").join(df_InvoiceAddendumDetails_shiptoAgg .alias("tgt"),((col("src.ULLogs_ShipToAccountId") == col("tgt.InvoiceAddendumDetails_ShipToAccountId")) & (col("src.ULLogs_CoolSculptingID") == col("tgt.InvoiceAddendumDetails_CoolSculptingID"))  & (col("src.ULLogs_HdrStartDateGeneratedDt") == col("tgt.InvoiceAddendumDetails_CycleDate")) & (col("src.ULLogs_PromotionUUID") == col("tgt.InvoiceAddendumDetails_PromotionUUID"))),"FULL")\
                                .withColumn("Comments",when(((col("src.ULLogs_PlanType") == 'SmallTreatmentPlan') | (col("src.ULLogs_FlagType").isin('OffBoarding_NonP3_Treatment','OffBoarding_NonP3','TNC','Fraud')) | (col("src.ULLogs_CoolSculptingID") == 'MISSING')) & (col("tgt.InvoiceAddendumDetails_PatientClassificationName") == 'PerCycleEfficacyFail') & (col("tgt.InvoiceAddendumDetails_IncrementalInvoiceCharge") == 0), "Matched")
                                                        .when(((col("src.ULLogs_PlanType") != 'SmallTreatmentPlan') & ((~col("src.ULLogs_FlagType").isin('OffBoarding_NonP3_Treatment','OffBoarding_NonP3','TNC','Fraud')) | (col("src.ULLogs_FlagType").isNull())) & (col("src.ULLogs_CoolSculptingID") != 'MISSING')),"Cycle was not eligible for Per Cycle Calculation")
                                                        .when(col("src.ULLogs_ConsumerSubscriptionUUID").isNotNull(),"Matched")
                                                        .when((col("src.ULLogs_UL_CreatedDt").between(col("src.ULLogs_CalendarEndTimeStamp") - expr(f"INTERVAL {BufferTime} MINUTES"),col("src.ULLogs_CalendarEndTimeStamp"))),"Cycle Received within 20 minutes of Billing Period End Date")
                                                        .when((col("tgt.InvoiceAddendumDetails_Details_CreatedDate").between(col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp"),col("tgt.InvoiceAddendumDetails_CalendarStartTimeStamp") + expr(f"INTERVAL {BufferTime} MINUTES"))),"Cycle Received within 20 minutes of Billing Period Start Date")
                                                        .otherwise("Unknown"))
    
    ExceptionCount = df_result.filter(col("Comments")== "Unknown").count() 
    MismatchedData = df_result.filter("Comments == 'Unknown'").toJSON().collect()
    
    runAssert(ExceptionCount == 0,TestCaseDetails[TestCaseName]["ErrorMessage"],TestCaseName,InvoiceDate,str(MismatchedData))
    print(f"PerCycleEfficacyFail Validation:")
    df_result.filter("Comments == 'Unknown'").display()